# READ (Scan) Optimizations
Reduce the amount of data loaded from storage into Spark.
| # | Optimization               | What It Skips                                      | Automatic?                               |
|---|---------------------------|----------------------------------------------------|-------------------------------------------|
| 1 | Partition Pruning         | Entire folders/partitions                          | Yes (if filter on partition column)       |
| 2 | Dynamic Partition Pruning | Folders, using filter values from a join          | Yes (Spark 3.0+)                          |
| 3 | Predicate Pushdown        | Rows at the scan layer                             | Yes (unless a UDF blocks pushdown)        |
| 4 | Column Pruning            | Unneeded columns in Parquet/ORC                    | Yes (if you avoid `SELECT *`)             |
| 5 | Data Skipping (min/max)   | Files whose value ranges don't match               | Yes (Delta)                               |


##### **1. Partition Pruning**

**Problem**
By default, Spark reads ALL files in a table — even if the query only needs a small subset.  
A table with 365 daily partitions forces Spark to list and open files in all 365 folders even when you want just one day.

**Concept**
When a table is partitioned (data stored in column=value/ folder structure),
Spark can use filters on the partition column to SKIP ENTIRE FOLDERS.
Spark resolves the filter at plan time, prunes non-matching partition
directories, and only opens files in matching folders.

Internally:
  - /data/sales/country=US/part-00000.parquet
  - /data/sales/country=UK/part-00000.parquet
  - /data/sales/country=IN/part-00000.parquet

Query: WHERE country = 'US'
- Spark reads ONLY /data/sales/country=US/  → skips UK and IN folders entirely.

**Solution**
Filter directly on the partition column so Spark can prune partitions during the scan.

**Key Points**

DO:
  - Partition on LOW-CARDINALITY, FREQUENTLY-FILTERED columns (date, region,
    status, country).  Ideal: 100s to low 1000s of distinct values.
  - Filter DIRECTLY on the partition column: WHERE date = '2025-01-01'
  - Verify pruning with .explain(True) — look for "PartitionFilters".
  - Combine with predicate pushdown on non-partition columns for max effect.

DON'T:
  - Partition on HIGH-CARDINALITY columns (user_id, transaction_id) —
    creates millions of tiny folders/files (small file problem).
  - Wrap the partition column in functions:
      WRONG:  WHERE year(date) = 2025        → full scan, no pruning
      RIGHT:  WHERE date >= '2025-01-01' AND date < '2026-01-01'
  - Use UDFs on partition columns in the filter — Spark can't push UDFs down.
  - Over-partition: more than ~10,000 partitions causes metadata overhead.

WHEN TO USE:
  - Tables > 1 GB that are repeatedly filtered on the same column(s).
  - ETL pipelines where new data arrives in time-based batches.
  - NOT useful if queries never filter on the partition column.

**Implementation**
```
def demo_partition_pruning():
    """Demonstrates partition pruning with and without proper filtering."""

    # --- Setup: Create a partitioned table ---
    data = [
        ("US", "2025-01-01", 100.0),
        ("US", "2025-01-02", 150.0),
        ("UK", "2025-01-01", 200.0),
        ("UK", "2025-01-02", 250.0),
        ("IN", "2025-01-01", 80.0),
        ("IN", "2025-01-02", 90.0),
    ]
    schema = StructType([
        StructField("country", StringType()),
        StructField("date", StringType()),
        StructField("revenue", DoubleType()),
    ])
    df = spark.createDataFrame(data, schema)
    df.write.mode("overwrite").partitionBy("country").parquet("/tmp/training/sales_partitioned")

    # --- GOOD: Filter on partition column → Spark prunes partitions ---
    df_pruned = spark.read.parquet("/tmp/training/sales_partitioned") \
        .filter("country = 'US'")

    print("=== GOOD: Partition Pruning Active ===")
    df_pruned.explain(True)
    # Look for: PartitionFilters: [isnotnull(country), (country = US)]
    # Spark reads ONLY the country=US folder.

    # --- BAD: UDF on partition column → Spark CANNOT prune ---
    from pyspark.sql.functions import udf

    @udf(StringType())
    def upper_country(c):
        return c.upper() if c else None

    df_no_prune = spark.read.parquet("/tmp/training/sales_partitioned") \
        .filter(upper_country(F.col("country")) == "US")

    print("\n=== BAD: UDF blocks partition pruning — full scan ===")
    df_no_prune.explain(True)
    # PartitionFilters will be empty — Spark scans ALL partitions.

    # --- BAD: Function wrapping partition column ---
    df_bad = spark.read.parquet("/tmp/training/sales_partitioned") \
        .filter(F.substring(F.col("country"), 1, 2) == "US")

    print("\n=== BAD: Function on partition column — full scan ===")
    df_bad.explain(True)
```


##### **2. Dynamic Partition Pruning (DPP)**
**Problem**
Standard partition pruning requires a LITERAL filter value in the query.
But often the filter comes from JOINING with another table:

```
    SELECT f.* FROM fact_sales f
    JOIN dim_date d ON f.date_key = d.date_key
    WHERE d.quarter = 'Q1-2025'
```

Without DPP, Spark performs a FULL SCAN of fact_sales because the filter
value isn't known until the dim_date side is evaluated.

**Concept**
Dynamic Partition Pruning (DPP) resolves the filter from the dimension
table FIRST, then pushes those values as a runtime filter into the fact
table scan.

Step-by-step:
  1. Spark evaluates: dim_date WHERE quarter = 'Q1-2025' → gets date_key list
  2. Spark injects those date_key values as a filter on the fact_sales scan
  3. fact_sales reads ONLY the matching date_key partitions

This happens automatically in Spark 3.0+ when conditions are met.

**Solution**
Ensure the fact table is partitioned on the join key, and the dimension table is small enough to be broadcast.

**Key Points**

DO:
  - Partition the FACT table on the join key (the column used in the ON clause).
  - Keep dimension tables small (broadcastable) — DPP works best when Spark
    can broadcast the dimension side.
  - Verify with .explain(True) — look for "DynamicPruningExpression".
  - Works automatically — no code change needed beyond proper partitioning.

DON'T:
  - Expect DPP to work if the fact table is NOT partitioned on the join key.
  - Disable it: spark.sql.optimizer.dynamicPartitionPruning.enabled = false.
  - Use DPP as a substitute for direct partition pruning — if you know the
    literal value, use it directly (faster plan, no dependency on dim query).

WHEN TO USE:
  - Star/snowflake schema queries: fact table joined with dimension tables.
  - Any join where one side is small and the other is large + partitioned.
  - NOT useful when both tables are large and neither is partitioned.

**Implementation**

```
def demo_dynamic_partition_pruning():
    """Demonstrates DPP with a fact-dimension join."""

    # --- Setup: Partitioned fact table ---
    fact_data = [
        ("2025-01-01", "prod_1", 100.0),
        ("2025-01-02", "prod_2", 200.0),
        ("2025-04-15", "prod_1", 300.0),
        ("2025-04-16", "prod_3", 400.0),
        ("2025-07-01", "prod_2", 500.0),
    ]
    fact_schema = StructType([
        StructField("date_key", StringType()),
        StructField("product_id", StringType()),
        StructField("revenue", DoubleType()),
    ])
    fact_df = spark.createDataFrame(fact_data, fact_schema)
    fact_df.write.mode("overwrite") \
        .partitionBy("date_key") \
        .parquet("/tmp/training/fact_sales")

    # --- Setup: Small dimension table ---
    dim_data = [
        ("2025-01-01", "Q1-2025"),
        ("2025-01-02", "Q1-2025"),
        ("2025-04-15", "Q2-2025"),
        ("2025-04-16", "Q2-2025"),
        ("2025-07-01", "Q3-2025"),
    ]
    dim_schema = StructType([
        StructField("date_key", StringType()),
        StructField("quarter", StringType()),
    ])
    dim_df = spark.createDataFrame(dim_data, dim_schema)

    # --- DPP in action ---
    # Spark resolves Q1-2025 dates from dim_df, then prunes fact_sales partitions
    result = spark.read.parquet("/tmp/training/fact_sales") \
        .join(F.broadcast(dim_df), "date_key") \
        .filter("quarter = 'Q1-2025'")

    print("=== Dynamic Partition Pruning ===")
    result.explain(True)
    # Look for: DynamicPruningExpression in the fact_sales scan
    result.show()
    
    ### Note: By default, DPP is enabled from Sprak 3.x and below spark 3 DPP won't exists. If someone disabled, use below code to enable it.
    spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning.enabled", "true")
```


##### **3. Predicate Pushdown**

**Problem**
- Without pushdown, Spark loads ALL rows from a file into memory, THEN applies the WHERE filter.  On a 10 GB Parquet file where only 1% of rows match, you waste 9.9 GB of I/O and memory.

**Concept**
- Predicate pushdown moves the WHERE filter DOWN INTO the data source reader.
- For Parquet/ORC, the filter is evaluated at the row-group level using min/max statistics — entire row groups are skipped without decoding rows.
- For JDBC sources, the filter becomes part of the SQL query sent to the database.

Pipeline WITHOUT pushdown: Read all rows → Load into memory → Apply filter → Output

Pipeline WITH pushdown: Read file metadata → Skip non-matching row groups → Decode only matching rows → Output

**Solution**
Use built-in Spark functions in filters.  Avoid Python UDFs, complex expressions, and non-deterministic functions in WHERE clauses.

**Key Points**

DO:
  - Use built-in functions: col("x") == "val", col("x").between(1, 10),
    col("x").isin(["a", "b"]), col("x").isNull().
  - Filter as EARLY as possible in the DataFrame chain.
  - Verify with .explain(True) — look for "PushedFilters:" in the Scan node.
  - For JDBC sources, pass filters via .option("pushDownPredicate", "true")
    and verify the query sent to the database.

DON'T:
  - Use Python UDFs in WHERE clauses — they are opaque to the optimizer;
    Spark cannot push them to the Parquet reader.
  - Use non-deterministic functions (rand(), **current_timestamp()**) in filters —
    Spark won't push these down since results can vary.
  - Assume complex expressions always push down — compound expressions like
    (col("a") + col("b") > 10) may not push depending on the source.
  - Filter AFTER a shuffle/join when you could have filtered BEFORE.

WHEN TO USE:
  - Always.  This is the single most impactful automatic optimization.
  - Especially important for large Parquet/ORC tables and JDBC reads.
  - Less relevant for CSV/JSON (no row-group stats to skip).

**Implementation**

```
def demo_predicate_pushdown():
    """Demonstrates predicate pushdown with Parquet files."""

    # --- Setup ---
    data = [(i, f"product_{i % 100}", float(i * 10), "active" if i % 3 == 0 else "inactive")
            for i in range(10000)]
    schema = StructType([
        StructField("id", IntegerType()),
        StructField("product", StringType()),
        StructField("amount", DoubleType()),
        StructField("status", StringType()),
    ])
    df = spark.createDataFrame(data, schema)
    df.write.mode("overwrite").parquet("/tmp/training/pushdown_demo")

    # --- GOOD: Built-in filter → pushed down to Parquet reader ---
    df_good = spark.read.parquet("/tmp/training/pushdown_demo") \
        .filter((F.col("status") == "active") & (F.col("amount") > 5000))

    print("=== GOOD: Predicate Pushdown Active ===")
    df_good.explain(True)
    # Look for: PushedFilters: [IsNotNull(status), EqualTo(status,active),
    #                           GreaterThan(amount,5000.0)]

    # --- BAD: Python UDF blocks pushdown ---
    from pyspark.sql.functions import udf

    @udf(StringType())
    def check_status(s):
        return "yes" if s == "active" else "no"

    df_bad = spark.read.parquet("/tmp/training/pushdown_demo") \
        .filter(check_status(F.col("status")) == "yes")

    print("\n=== BAD: UDF Blocks Pushdown — Full Scan ===")
    df_bad.explain(True)
    # PushedFilters will be EMPTY — every row is read then filtered in Python.

    # --- PATTERN: Filter BEFORE join, not after ---
    other_df = spark.range(100).withColumn("product", F.concat(F.lit("product_"), F.col("id")))

    # BAD: filter after join
    result_bad = spark.read.parquet("/tmp/training/pushdown_demo") \
        .join(other_df, "product") \
        .filter("status = 'active'")

    # GOOD: filter before join — fewer rows enter the shuffle
    result_good = spark.read.parquet("/tmp/training/pushdown_demo") \
        .filter("status = 'active'") \
        .join(other_df, "product")

    print("\n=== GOOD: Filter BEFORE join ===")
    result_good.explain(True)
```


##### **4. Column Pruning (Project Pushdown)**
**Problem**
- Using SELECT * on a wide table (100+ columns) forces Spark to read EVERY column from Parquet, even if the query only needs 2.  On columnar formats, this is a massive waste — Parquet stores each column in separate column chunks, so unneeded columns can be entirely skipped.

**Concept**
- Parquet/ORC files are columnar: data is stored column-by-column, not row-by-row.  When you SELECT only specific columns, Spark's Parquet reader skips the byte ranges of all other columns — never reads them from disk.

```
    Table: 200 columns, 1 TB total
    SELECT col_a, col_b → reads ~10 GB (2/200 columns ≈ 1% of data)
    SELECT *            → reads 1 TB
```
This is called PROJECTION PUSHDOWN — the column list (projection) is
pushed into the reader.

**Solution**
- Always select only the columns you need.  Never use SELECT * in production.

**Key Points**

DO:
  - Explicitly list columns: df.select("col_a", "col_b")
  - Select early in the chain — before joins, groupBys, or any transformation.
  - For nested structs, select only needed fields: df.select("address.city")
    (requires nested schema pruning — see section 5).
  - Use .drop() to remove a few unneeded columns from a mostly-needed schema.

DON'T:
  - Use SELECT * or df.select("*") in production code.
  - Read all columns then filter them down later in the chain — Spark's
    optimizer can sometimes push projections down, but don't rely on it
    for complex plans.
  - Assume CSV/JSON benefits equally — they are row-based; the entire row
    is read regardless (but Spark still avoids deserializing unused columns).

WHEN TO USE:
  - Always.  There is no reason not to prune columns.
  - Impact scales with table width: 10 columns → minor; 200 columns → massive.
"""

**Implementation**
```
def demo_column_pruning():
    """Demonstrates column pruning impact on Parquet reads."""

    # --- Setup: Wide table with many columns ---
    data = [(i,) + tuple(f"val_{j}_{i}" for j in range(20)) for i in range(1000)]
    columns = ["id"] + [f"col_{j}" for j in range(20)]
    df = spark.createDataFrame(data, columns)
    df.write.mode("overwrite").parquet("/tmp/training/wide_table")

    # --- BAD: SELECT * reads all 21 columns ---
    df_bad = spark.read.parquet("/tmp/training/wide_table")
    result_bad = df_bad.groupBy("col_0").count()

    print("=== BAD: SELECT * — reads all columns ===")
    result_bad.explain(True)

    # --- GOOD: Select only needed columns — reads 2 columns ---
    df_good = spark.read.parquet("/tmp/training/wide_table") \
        .select("id", "col_0")
    result_good = df_good.groupBy("col_0").count()

    print("\n=== GOOD: Column Pruning — reads only id and col_0 ===")
    result_good.explain(True)
    # In the Scan node, ReadSchema will show only the selected columns.
```


##### **5. Data Skipping (Min/Max Statistics)**
**Problem**
Even after partition pruning, Spark may still open EVERY file within a
partition.  A partition with 1,000 files must open all 1,000 even if only
1 file contains matching rows.

**Concept**
Parquet files store MIN/MAX statistics per column per ROW GROUP (typically
128 MB chunks).  Delta Lake extends this by storing per-FILE min/max in
the transaction log.

When a filter like WHERE id BETWEEN 100 AND 200 is applied:
  - Parquet reader checks each row group's min/max for column "id"
  - If a row group's range is [500, 1000], it doesn't overlap [100, 200]
    → entire row group is SKIPPED without reading any data
  - Delta Lake does the same at FILE level using the transaction log

EFFECTIVENESS depends on data ordering:
  - SORTED data → tight min/max ranges → excellent skipping
  - RANDOM data → wide min/max ranges → no skipping
    Example: File 1 has id [1, 100000] → overlaps almost any filter

**Solution**
For Parquet: Write data sorted by frequently-filtered columns.
For Delta: Use OPTIMIZE + ZORDER BY or Liquid Clustering.

**Key Points**

DO:
  - Sort data on the most frequently filtered column before writing.
  - For Delta tables, run OPTIMIZE with ZORDER BY on filter columns.
  - Place frequently filtered columns in the FIRST 32 positions — Delta
    collects stats only on the first 32 columns by default.
  - Use range filters (BETWEEN, >, <) — they benefit most from min/max.

DON'T:
  - Expect data skipping on randomly ordered data — min/max will span the
    full range of values, so no files can be skipped.
  - Rely on skipping for string columns with long values — stats are
    truncated (default 32 chars) and may not be useful.
  - Forget to run OPTIMIZE periodically — new appended files won't be
    co-located until compacted.
  - Put filter columns beyond position 32 in the schema (for Delta).

WHEN TO USE:
  - Large Delta tables with range filters on ordered/clustered columns.
  - Timestamp, date, ID columns with natural ordering.
  - Less useful for boolean or low-cardinality columns (min=false, max=true
    covers everything).

**Implementation**
```
def demo_data_skipping():
    """Demonstrates data skipping with sorted vs. unsorted data."""

    # --- Setup: Generate data ---
    from pyspark.sql.functions import rand

    df = spark.range(0, 100000).withColumn("value", (F.col("id") * 3.14).cast("double"))

    # --- Sorted write: tight min/max per file → good skipping ---
    df.orderBy("id").repartition(10).write.mode("overwrite") \
        .parquet("/tmp/training/sorted_data")

    # --- Random write: wide min/max per file → poor skipping ---
    df.orderBy(rand()).repartition(10).write.mode("overwrite") \
        .parquet("/tmp/training/random_data")

    # --- Query both with a range filter ---
    print("=== Sorted Data: Data Skipping Effective ===")
    df_sorted = spark.read.parquet("/tmp/training/sorted_data") \
        .filter("id BETWEEN 100 AND 200")
    df_sorted.explain(True)
    # PushedFilters will show the range predicate.
    # On sorted data, most files will be skipped via min/max.

    print("\n=== Random Data: Data Skipping Ineffective ===")
    df_random = spark.read.parquet("/tmp/training/random_data") \
        .filter("id BETWEEN 100 AND 200")
    df_random.explain(True)
    # Same pushed filter, but ALL files will be opened because
    # each file's min/max range spans nearly [0, 100000].

    # --- Delta Lake ZORDER example (requires Delta) ---
    # Uncomment if running on Databricks or with delta-spark:
    #
    # df.write.mode("overwrite").format("delta").save("/tmp/training/delta_table")
    # spark.sql("OPTIMIZE delta.`/tmp/training/delta_table` ZORDER BY (id)")
    # # Now id values are co-located → tight min/max → excellent skipping
```


# TRANSFORMATION (Compute) Optimizations
Make operations like joins, aggregations, and UDFs faster.

| #  | Optimization                      | Category        | What It Fixes                                      |
|----|-----------------------------------|-----------------|----------------------------------------------------|
|1| AQE — Join Conversion             | Join            | Switches Sort-Merge → Broadcast at runtime         |
|2| AQE — Skew Handling               | Join / Skew     | Splits skewed partitions into smaller tasks        |
|3| AQE — Coalescing Partitions       | Shuffle         | Merges too-many small post-shuffle partitions      |
|4| Broadcast Join                    | Join            | Eliminates shuffle by broadcasting small table     |
|5| Salting                           | Join / Skew     | Breaks hot keys into sub-keys for even distribution|
|6| Shuffle Partitions Tuning         | Shuffle         | Sets right partition count for data size           |
|7| Repartition                       | Shuffle         | Redistributes data evenly or by key                |
|8| Coalesce                          | Shuffle         | Reduces partitions without full shuffle            |

##### 1. AQE — JOIN CONVERSION (Sort-Merge → Broadcast at Runtime)
**Problem**
At PLAN TIME, Spark estimates table sizes using catalog statistics or file
sizes.  If stats are missing, stale, or inaccurate, Spark may choose a
Sort-Merge Join (SMJ) for a join where one side is actually small enough
to broadcast — wasting time on an expensive shuffle of BOTH sides.

Example:
-   Table A: catalog says 500 MB  →  Spark picks Sort-Merge Join
-   Reality: after filter, Table A is only 8 MB  →  should be Broadcast

Without AQE: both tables get shuffled (network transfer + disk I/O).
With AQE: after the filter stage completes, Spark sees the REAL size is
8 MB and SWITCHES to Broadcast Hash Join — no shuffle on the large side.

**Concept**

Adaptive Query Execution (AQE) re-optimizes the plan BETWEEN STAGES.
After each stage completes, Spark has ACTUAL data sizes (not estimates).

For join conversion:
  1. Stage 1 (filter + scan) completes → Spark sees actual output = 8 MB
  2. Spark checks: 8 MB < autoBroadcastJoinThreshold (default 10 MB)?
  3. YES → switches from Sort-Merge Join to Broadcast Hash Join
  4. Large table side skips the shuffle entirely

This is the MOST IMPACTFUL AQE feature because it eliminates shuffles
that were planned based on wrong estimates.

**Solution**
Enable AQE (default in Spark 3.2+).  Ensure autoBroadcastJoinThreshold
is set appropriately.

**Key Points**

DO:
  - Keep AQE enabled: spark.sql.adaptive.enabled = true (default in 3.2+).
  - Set a reasonable autoBroadcastJoinThreshold:
      Default: 10 MB.  Increase to 50-100 MB if executors have enough memory.
  - Filter tables BEFORE joins — AQE can only convert AFTER seeing the
    filtered size.  If you filter after the join, AQE sees the full size.
  - Run ANALYZE TABLE to give Spark better initial estimates — AQE then
    confirms/overrides at runtime.
  - Verify with .explain() or Spark UI SQL tab — look for
    "BroadcastHashJoin" instead of "SortMergeJoin".

DON'T:
  - Disable AQE unless you have a specific Spark 2.x compatibility need.
  - Set autoBroadcastJoinThreshold too high (> 1 GB) — broadcasting a large
    table can cause driver/executor OOM.
  - Assume AQE always converts — it only converts when the ACTUAL post-stage
    size is below the threshold.  If both sides are large, SMJ is correct.
  - Rely solely on AQE without ANALYZE TABLE — AQE corrects bad plans at
    runtime, but good initial stats = better plan from the start.

WHEN TO USE:
  - Always enabled.  Zero downside in Spark 3.2+.
  - Most impactful when tables are heavily filtered before joins (real size
    is much smaller than raw table size).
  - Less impactful when both sides of the join are genuinely large.
"""

**Implementation:**

```
# Enable Adaptive Query Execution (AQE)
spark.conf.set("spark.sql.adaptive.enabled", "true")

# Set auto broadcast join threshold (in bytes)
# Example: 10 MB
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)

# Optional: increase timeout
spark.conf.set("spark.sql.broadcastTimeout", 300)

# For Disable
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

# Check Broadcast in Execution Plan
df.join(df2, "id").explain(True) 
Look for BroadcastHashJoin
```

##### **2. AQE — SKEW HANDLING**
**Problem**

Data skew means one join key has disproportionately more rows than others.

    key = "US"  →  50 million rows  (one task handles ALL of these)
    key = "UK"  →  500K rows
    key = "IN"  →  800K rows

In a Sort-Merge Join, all rows with the same key go to the SAME task.
The "US" task runs for hours while all other tasks finish in seconds.
This is called a STRAGGLER TASK — one slow task blocks the entire stage.

Worse: the "US" task may run out of memory (OOM) because 50M rows don't
fit in one executor's memory.

**Concept**

AQE detects skewed partitions AFTER the shuffle stage completes (it can
see the actual partition sizes).  When a partition is significantly larger
than the median:

  1. AQE identifies the skewed partition (e.g., key = "US", 50M rows)
  2. It SPLITS the skewed partition into multiple smaller sub-partitions
  3. Each sub-partition is joined with a COPY of the matching data from
     the other side
  4. Results are combined — same output, but work is distributed

Detection formula:
  Partition is skewed if:
    partition_size > skewedPartitionFactor × median_partition_size
    AND partition_size > skewedPartitionThresholdInBytes

**Soultion**

Enable AQE with skew join handling (default in Spark 3.2+).

**Key Points**

DO:
  - Keep AQE skew handling enabled:
      spark.sql.adaptive.skewJoin.enabled = true (default).
  - Tune detection thresholds for your data:
      skewedPartitionFactor = 5 (default) — partition must be 5x median
      skewedPartitionThresholdInBytes = 256MB (default) — minimum size to split
  - Check Spark UI SQL tab for "skew join optimization" in the plan.
  - Use AQE as the FIRST defense against skew — it's automatic.

DON'T:
  - Rely on AQE alone for extreme skew (1000:1 ratio) — manual salting
    may still be needed.
  - Set skewedPartitionFactor too low (e.g., 1.5) — it will split
    partitions that aren't truly skewed, adding overhead.
  - Set skewedPartitionThresholdInBytes too low — small partitions will be
    split unnecessarily.
  - Confuse AQE skew handling with salting — AQE is automatic at runtime;
    salting is a manual code technique.

WHEN TO USE:
  - Always enabled as a safety net.
  - Detects and fixes moderate skew automatically.
  - For extreme/known skew, combine with manual salting (see section 5).

**Implementation**

```
# Enable AQE (mandatory)
spark.conf.set("spark.sql.adaptive.enabled", "true")

# Enable skew join handling
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

# If a partition is 5x larger than median → considered skewed
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "5")

# Minimum size to consider skew (default ~256MB)
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", 256 * 1024 * 1024)
```

##### **3. AQE — COALESCING POST-SHUFFLE PARTITIONS**
**Problem**
Spark's default shuffle creates 200 partitions:
```
  spark.sql.shuffle.partitions = 200 (default)
```

- After a shuffle (join, groupBy, window), data is redistributed into 200 partitions. If the data is small (e.g., 50 MB total), you get: 200 partitions × 250 KB each = 200 tiny tasks
- Each task has scheduling overhead (~50-100ms), so 200 tasks that each do 100ms of work spend more time on scheduling than on actual processing.
- The reverse: if data is 500 GB, 200 partitions means 2.5 GB per partition — too large, may OOM.

**Concept**

AQE coalescing automatically MERGES small post-shuffle partitions into larger ones AFTER the shuffle completes (when actual sizes are known).

Process:
  1. Shuffle writes 200 partitions
  2. AQE checks actual sizes of each partition
  3. Adjacent small partitions are merged until they reach the advisory
     target size (default 64 MB)
  4. 200 tiny partitions → maybe 5 appropriately-sized partitions

This solves the "too many small partitions" problem WITHOUT you having to guess the right spark.sql.shuffle.partitions value upfront.

**Solution**

Enable AQE coalescing (default in Spark 3.2+).

**Key Points**

DO:
  - Keep AQE coalescing enabled:
      spark.sql.adaptive.coalescePartitions.enabled = true (default).
  - Set the advisory partition size:
      spark.sql.adaptive.advisoryPartitionSizeInBytes = 64MB (default)
      Increase to 128-256 MB for large datasets.
  - Still set spark.sql.shuffle.partitions to a reasonable UPPER BOUND
    (e.g., 200 or 2000) — AQE can only REDUCE, never increase.
  - Verify in Spark UI: check "number of partitions" in the stage after shuffle.

DON'T:
  - Rely on AQE to increase partitions — it only coalesces (merges).
    If you need MORE partitions, increase spark.sql.shuffle.partitions.
  - Set advisoryPartitionSizeInBytes too large (> 512 MB) — tasks may
    run out of memory.
  - Set spark.sql.shuffle.partitions = 1 thinking AQE will fix it —
    AQE can only merge, not split.

WHEN TO USE:
  - Always enabled.  Handles the common case of too many small partitions.
  - Especially useful when the same Spark job processes varying data sizes
    (some runs have 10 MB, others have 100 GB).

**Implementation**
```
# Enable AQE (mandatory)
spark.conf.set("spark.sql.adaptive.enabled", "true")

# Enable coalescing of partitions
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

# Target size of each partition after coalescing (default ~64MB)
spark.conf.set(
    "spark.sql.adaptive.advisoryPartitionSizeInBytes",
    64 * 1024 * 1024
)
```

##### 4. Broadcast Join
**Problem**

In join operations, Spark typically performs a **shuffle join** (e.g., Sort-Merge Join):

- Both tables are shuffled across the cluster
- Data movement is expensive (network + disk I/O)
- Performance degrades for large datasets

Even if one table is small, Spark may still choose a shuffle join depending on configs.

**Concept**

A **Broadcast Join** avoids shuffle by:

1. Sending (broadcasting) the **small table** to all executors
2. Each executor joins its partition of the large table locally

👉 No shuffle required  
👉 Much faster for small + large table joins  

**Solution**

Manually force broadcast using `broadcast()`

**Key Points**

DO:

- Use when one table is small (few MBs to ~100MB depending on cluster)
- Use for dimension tables in fact-dimension joins
- Use when Spark is not automatically choosing broadcast
- Verify using .explain(True) → look for BroadcastHashJoin

DON'T:

- Don’t broadcast large tables → can cause OOM
- Don’t blindly broadcast without checking size
- Don’t use if both tables are large

When to Use

- Fact (large) + Dimension (small) joins
- Lookup tables
- Enrichment joins

**Implementation**

```
from pyspark.sql.functions import broadcast

result = large_df.join(
    broadcast(small_df),
    "join_key"
) 
```

##### **5. SALTING (Manual Skew Fix)**

**Problem**
One join key has disproportionately more rows than others (data skew). AQE skew handling helps, but for EXTREME skew (1000:1 ratio) or when you KNOW which keys are hot, manual salting gives more control.

    - key = "US"   →  50 million rows (one executor overwhelmed → OOM)
    - key = "UK"   →  500K rows
    - key = "IN"   →  800K rows

The executor handling "US" runs out of memory or takes hours while all other tasks finish in seconds.

**Concept**

Salting is a **manual technique** to distribute skewed data *before shuffle*.

👉 Idea:
Break one large key into multiple smaller keys

Step-by-step:
  1. Add a random salt (0 to N-1) to the LARGE table's join key:
     "US" → "US_0", "US_1", ..., "US_9"   (10 sub-partitions)
  2. EXPLODE the SMALL table to match ALL salt values:
     "US" → "US_0", "US_1", ..., "US_9"   (10 copies of lookup row)
  3. Join on the salted key → data is evenly distributed
  4. Drop the salt columns

BEFORE:  1 task handles 50M rows for "US"
AFTER:   10 tasks each handle 5M rows for "US_0" through "US_9"

**Solution**

Add salt to the large table, explode the small table, join on salted key.

**Key Points**

DO:
  - Choose salt count based on skew ratio:
      10 salts for 10:1 skew, 100 for 100:1, etc.
  - Salt ONLY the skewed keys if you know them — don't salt everything.
  - Use AQE as the first line of defense; add salting only when AQE
    isn't enough (extreme skew, known hot keys).
  - Remove salt columns after the join to keep output clean.

DON'T:
  - Over-salt — 1000 salts on moderately skewed data creates overhead
    (exploding the small table 1000x).
  - Forget to explode the small table — if you only salt the large side,
    keys won't match.
  - Use salting when broadcast join works — if the small table fits in
    memory, broadcast eliminates the problem entirely.
  - Apply salting blindly — profile your data first to identify which
    keys are actually skewed.

WHEN TO USE:
  - Known hot keys with extreme skew that AQE can't fully resolve.
  - Both tables are too large to broadcast.
  - After confirming skew in Spark UI (one task 100x slower than median).

**Implementation**

```
def demo_salting():
    """Demonstrates salting technique for skewed joins."""

    from pyspark.sql import functions as F
    from pyspark.sql.functions import explode, array, lit

    # --- Setup: Skewed large table (90% hot_key) ---
    skewed_data = (
        [(i, "hot_key", float(i)) for i in range(90000)] +
        [(i, f"key_{i % 100}", float(i)) for i in range(90000, 100000)]
    )
    large_df = spark.createDataFrame(skewed_data, ["id", "join_key", "amount"])

    # Small lookup table
    lookup_data = [("hot_key", "Hot Category", 1.5)] + \
                  [(f"key_{i}", f"Category_{i}", float(i)/100) for i in range(100)]
    small_df = spark.createDataFrame(lookup_data, ["join_key", "category", "multiplier"])

    # --- BAD: Direct join ---
    print("=== BAD: Direct join (skew issue) ===")
    result_bad = large_df.join(small_df, "join_key")
    result_bad.explain(True)

    # --- GOOD: Salted join ---
    num_salts = 10

    # Step 1: Salt large table
    large_salted = large_df.withColumn(
        "salt", F.floor(F.rand() * num_salts).cast("int")
    ).withColumn(
        "salted_key", F.concat_ws("_", F.col("join_key"), F.col("salt"))
    )

    # Step 2: Expand small table
    salt_array = array([lit(i) for i in range(num_salts)])

    small_exploded = small_df.withColumn(
        "salt", explode(salt_array)
    ).withColumn(
        "salted_key", F.concat_ws("_", F.col("join_key"), F.col("salt"))
    )

    # Step 3: Join
    result_good = large_salted.join(small_exploded, "salted_key") \
        .drop("salt", "salted_key")

    print("\n=== GOOD: Salted join ===")
    result_good.explain(True)

    # Validate
    print(f"Direct join count: {result_bad.count()}")
    print(f"Salted join count: {result_good.count()}")
```

##### 6. Shuffle Partitions Tuning

**Problem**

Every shuffle (join, groupBy, window, distinct, repartition) redistributes data into spark.sql.shuffle.partitions number of partitions.  Default: 200.

  TOO FEW partitions (200 for 500 GB data):
    500 GB / 200 = 2.5 GB per partition → OOM, disk spills
    Few tasks → under-utilizes cluster cores

  TOO MANY partitions (200 for 50 MB data):
    50 MB / 200 = 250 KB per partition → scheduling overhead
    200 tiny tasks → more time scheduling than processing

**Concept**

This setting controls the number of partitions AFTER any shuffle operation. It does NOT affect input scan partitions (those are controlled by spark.sql.files.maxPartitionBytes).

Ideal partition size: 100-200 MB after shuffle.

    Data size after shuffle    Recommended partitions
    ─────────────────────────  ─────────────────────
    < 1 GB                     10-50
    1-10 GB                    50-200
    10-100 GB                  200-2000
    100 GB - 1 TB              2000-10000
    > 1 TB                     10000+

With AQE coalescing enabled, set this to an UPPER BOUND — AQE will merge
small partitions down.  Without AQE, you must guess the right value.

**Solution**

Set based on your typical data size.  With AQE, err on the high side.

**Key Points**

DO:
  - With AQE: set to a generous upper bound (e.g., 2000 for multi-GB data).
    AQE will coalesce down to the right number.
  - Without AQE: calculate → total_shuffle_data / 128MB = partition_count.
  - Set per-job if data sizes vary:
      spark.conf.set("spark.sql.shuffle.partitions", "500")
  - Monitor in Spark UI: check partition sizes in the stage after shuffle.
    Target: 100-200 MB per partition.

DON'T:
  - Leave at 200 for large datasets (> 50 GB) — too few, causes OOM/spills.
  - Leave at 200 for small datasets (< 100 MB) — too many, wastes scheduling.
  - Set to 1 — all data goes to one partition (guaranteed OOM on large data).
  - Confuse with spark.sql.files.maxPartitionBytes (input scan) or
    df.repartition() (explicit repartition).

WHEN TO USE:
  - Every Spark job.  This is the most commonly misconfigured setting.
  - Adjust based on your data size and cluster resources.

**Implementation**

```
# -------------------------------
# Baseline: Set shuffle partitions
# -------------------------------
# Keep this reasonably high to allow parallelism
spark.conf.set("spark.sql.shuffle.partitions", 200)

# -------------------------------
# Enable AQE (Adaptive Query Execution)
# -------------------------------
spark.conf.set("spark.sql.adaptive.enabled", "true")

# -------------------------------
# Enable AQE Coalesce (reduce small partitions)
# -------------------------------
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

# Target partition size after coalescing (default ~64MB)
spark.conf.set(
    "spark.sql.adaptive.advisoryPartitionSizeInBytes",
    64 * 1024 * 1024
)

# -------------------------------
# Enable AQE Skew Join Handling
# -------------------------------
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

# Skew detection configs (optional tuning)
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "5")
spark.conf.set(
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes",
    256 * 1024 * 1024
)

# -------------------------------
# (Optional) Enable dynamic join conversion
# -------------------------------
# Allows Spark to convert sort-merge join → broadcast join at runtime
spark.conf.set("spark.sql.adaptive.join.enabled", "true")

```

##### **7. Repartition**

**Problem**

Data may be unevenly distributed across partitions after a read or transformation:
  - Partition 1: 500 MB (overloaded)
  - Partition 2: 10 MB
  - Partition 3: 5 MB

Or you need to redistribute data BY KEY before a join to avoid a shuffle
at join time.

Or you need MORE partitions to utilize all available cores (input scan
created 4 partitions but you have 100 cores).

**Concept**

repartition(n) creates EXACTLY n partitions by doing a FULL SHUFFLE — all data is redistributed evenly using round-robin or hash partitioning.

Two modes:
  - repartition(n)           → round-robin: even distribution, random order
  - repartition(n, "col")    → hash: rows with same col value go to same partition
  - repartition("col")       → hash with default partition count

FULL SHUFFLE means ALL data moves across the network.  This is expensive but sometimes necessary.

**Solution**

Use repartition to increase partitions, redistribute evenly, or organize by key.

**Key Points**

DO:
  - repartition(n) to INCREASE partition count (coalesce can only decrease).
  - repartition("join_key") BEFORE a join to pre-shuffle by the join key —
    can eliminate the shuffle at join time if both sides are repartitioned.
  - repartition(n) before write to control output file count.
  - Use when you need EVEN distribution (after a skew-inducing operation).

DON'T:
  - Use repartition to DECREASE partitions — use coalesce instead
    (no shuffle needed to merge adjacent partitions).
  - Repartition unnecessarily — it's a full shuffle.  Only use when the
    redistribution provides a clear downstream benefit.
  - Repartition by a high-cardinality column with low partition count —
    e.g., repartition(5, "user_id") with 1M users → extreme hash collisions.
  - Repartition right before .count() or .show() — no benefit, just overhead.

WHEN TO USE:
  - Increasing parallelism: 4 input partitions → repartition(100) for 100 cores.
  - Pre-partitioning for join: both sides repartitioned by join key.
  - Controlling output files: repartition(10).write.parquet() → 10 files.
  - Fixing skew: uneven partitions → repartition(n) for even distribution.

**Implementation**
```
def demo_repartition():
    """Demonstrates repartition use cases."""

    df = spark.range(0, 100000) \
        .withColumn("key", (F.col("id") % 100).cast("string")) \
        .withColumn("value", (F.col("id") * 2.5).cast("double"))

    # --- Check current partitions ---
    print(f"=== Initial partitions: {df.rdd.getNumPartitions()} ===")

    # --- repartition(n): Even distribution (round-robin) ---
    df_even = df.repartition(20)
    print(f"After repartition(20): {df_even.rdd.getNumPartitions()} partitions")

    # --- repartition("col"): Hash by column ---
    df_by_key = df.repartition("key")
    print(f"After repartition('key'): {df_by_key.rdd.getNumPartitions()} partitions")
    # Same key always goes to same partition — useful before joins.

    # --- repartition(n, "col"): Hash by column with specific count ---
    df_by_key_10 = df.repartition(10, "key")
    print(f"After repartition(10, 'key'): {df_by_key_10.rdd.getNumPartitions()} partitions")

    # --- Pre-join repartitioning to avoid shuffle at join time ---
    df1 = spark.range(100000).withColumn("join_key", (F.col("id") % 100).cast("int"))
    df2 = spark.range(1000).withColumnRenamed("id", "join_key")

    # Pre-repartition both by join key
    df1_repartitioned = df1.repartition("join_key")
    df2_repartitioned = df2.repartition("join_key")
    result = df1_repartitioned.join(df2_repartitioned, "join_key")

    print("\n=== Pre-repartitioned Join ===")
    result.explain(True)
```

##### 8. **Coalesce**
**Problem**

After filtering or other operations, you may have too many partitions with very little data in each:

    Before filter:  200 partitions × 100 MB each = 20 GB
    After filter:   200 partitions × 500 KB each = 100 MB total

200 partitions for 100 MB = 200 tiny tasks = scheduling overhead.

Or: before writing, you want fewer output files — 200 partitions would
create 200 tiny files (small file problem).

**Concept**

coalesce(n) REDUCES partition count by merging adjacent partitions. It does NOT shuffle data — partitions are simply combined locally.

    coalesce(10):  merge partitions [1-20] → partition 1
                   merge partitions [21-40] → partition 2
                   ...

Because there's NO SHUFFLE, coalesce is much faster than repartition.
But the resulting partitions may be UNEVEN — if partition 1 had 500 MB and
partition 2 had 5 MB, merging them gives 505 MB (no rebalancing).

**Solution**

Use coalesce to reduce partitions before writes or when data shrinks after filtering.

**Key Points**

DO:
  - Use coalesce to REDUCE partitions (e.g., 200 → 10 before write).
  - Use before write to control output file count:
      df.coalesce(10).write.parquet("/output/")  → 10 files
  - Use after a filter that dramatically reduces data size.
  - Prefer coalesce over repartition when decreasing — no shuffle.

DON'T:
  - Use coalesce to INCREASE partitions — coalesce(100) on a 10-partition
    DataFrame gives 10 partitions (it cannot split, only merge).
  - Use coalesce when you need EVEN distribution — coalesce doesn't
    rebalance, so output partitions may be very uneven.
  - coalesce(1) on large data — creates one giant file/partition,
    no parallelism on read.

WHEN TO USE:
  - Reducing output file count before write.
  - After heavy filtering (200 → 10 partitions).
  - When you need fewer partitions WITHOUT the cost of a full shuffle.
  - NOT when you need even distribution (use repartition instead).

**INTERVIEW TRAP:**
  "Can you use coalesce to increase partitions?"
  - ANSWER: No. coalesce can only reduce. Use repartition to increase.

  "What's the difference between repartition and coalesce?"
  - ANSWER: repartition = full shuffle (can increase or decrease, even distribution)
          - coalesce = no shuffle (can only decrease, may be uneven)

**Implementation**

```
def demo_coalesce():
    """Demonstrates coalesce for reducing partitions without shuffle."""

    df = spark.range(0, 100000) \
        .withColumn("value", (F.col("id") * 2.5).cast("double"))

    # --- Start with many partitions ---
    df_many = df.repartition(200)
    print(f"=== Starting partitions: {df_many.rdd.getNumPartitions()} ===")

    # --- coalesce: reduce without shuffle ---
    df_coalesced = df_many.coalesce(10)
    print(f"After coalesce(10): {df_coalesced.rdd.getNumPartitions()} partitions")

    # --- Verify: no Exchange (shuffle) node in plan ---
    print("\n=== coalesce(10) plan — NO Exchange node ===")
    df_coalesced.explain(True)
    # No "Exchange" node — coalesce is purely local merging.

    # --- Compare: repartition(10) adds a shuffle ---
    df_repartitioned = df_many.repartition(10)
    print("\n=== repartition(10) plan — HAS Exchange node ===")
    df_repartitioned.explain(True)
    # "Exchange RoundRobinPartitioning(10)" — full shuffle.

    # --- TRAP: coalesce cannot increase ---
    df_small = spark.range(100).repartition(5)
    df_try_increase = df_small.coalesce(20)
    print(f"\n=== TRAP: coalesce(20) on 5-partition DF → {df_try_increase.rdd.getNumPartitions()} partitions ===")
    # Still 5 — coalesce cannot increase.

    # --- Common pattern: coalesce before write ---
    print("\n=== Pattern: coalesce before write ===")
    # df.filter("status = 'active'").coalesce(10).write.parquet("/output/")
    print("df.filter(...).coalesce(10).write.parquet('/output/')")
    print("Creates exactly 10 output files instead of 200 tiny files.")
```

# WRITE & STORAGE Optimizations
Control output file layout for future read performance.

| # | Optimization               | When to Use                                     |
|---|---------------------------|-------------------------------------------------|
| 1 | Coalesce before write     | Reduce output files (no shuffle)                |
| 2 | Repartition before write  | Even file sizes or increase file count          |
| 3 | partitionBy("col")        | Enable partition pruning for downstream reads   |
| 4 | optimizeWrite (Delta)     | Auto-size files at write time                   |
| 5 | autoCompact (Delta)       | Auto-compact after frequent appends             |
| 6 | OPTIMIZE (Delta)          | Compact accumulated small files                 |
| 7 | ZORDER BY (Delta)         | Co-locate data for better data skipping         |
| 8 | V-Order (Fabric)          | Faster reads in Fabric engine                   |
| 9 | VACUUM (Delta)            | Remove old files and reclaim storage            |

##### 1. Coalesce Before Write
**Problem**

After a shuffle or filter, your DataFrame may have 200 partitions with tiny data in each.  Each partition becomes ONE output file:

    200 partitions → 200 files × 500 KB each = 100 MB in 200 tiny files

Downstream readers suffer:
  - Slow file listing (200 LIST calls on cloud storage)
  - 200 tasks for 100 MB of data (scheduling overhead)
  - Poor compression (small files can't compress efficiently)

**Concept**

coalesce(n) merges adjacent partitions into fewer, larger partitions. WITHOUT a shuffle.  Fewer partitions = fewer output files.

    df has 200 partitions → coalesce(10) → 10 partitions → write → 10 files

No data moves across the network — partitions are merged locally on the same executor.

**Solution**

Add coalesce(n) just before .write to control output file count.

Key Points

DO:
  - Target output file size: 128 MB – 1 GB per file.
  - Calculate: total_data_size / target_file_size = number of files.
    Example: 5 GB data / 500 MB target = 10 files → coalesce(10)
  - Use AFTER filter/transformation that reduces data significantly.

DON'T:
  - coalesce(1) on large data — one giant file, no read parallelism.
  - Use coalesce when data is UNEVENLY distributed — merged partitions
    will be uneven too.  Use repartition instead for even files.
  - Coalesce to MORE partitions than you have — it can only reduce.

**Implementation**

```
def demo_coalesce_before_write():
    """Coalesce reduces output file count without shuffle."""

    df = spark.range(0, 100000) \
        .withColumn("name", F.concat(F.lit("user_"), F.col("id").cast("string"))) \
        .withColumn("amount", (F.col("id") * 1.5).cast("double"))

    print(f"Partitions before coalesce: {df.rdd.getNumPartitions()}")

    # --- BAD: Many tiny output files ---
    df.write.mode("overwrite").parquet("/tmp/training/write_no_coalesce")

    # --- GOOD: Controlled file count ---
    df.coalesce(5).write.mode("overwrite").parquet("/tmp/training/write_coalesced")

    # Count output files
    import os
    bad_files = spark.read.parquet("/tmp/training/write_no_coalesce").inputFiles()
    good_files = spark.read.parquet("/tmp/training/write_coalesced").inputFiles()
    print(f"Without coalesce: {len(bad_files)} files")
    print(f"With coalesce(5): {len(good_files)} files")
```

##### **2. Repartition Before Write**

**Problem**

coalesce produces UNEVEN files because it merges without rebalancing. If partition 1 has 500 MB and partition 2 has 5 MB, coalescing them gives 505 MB — one huge file and one tiny file.

Also, you may need to INCREASE file count (coalesce can only decrease).

**Concept**

repartition(n) does a FULL SHUFFLE to create exactly n EVENLY-SIZED
partitions.  Every output file will be roughly the same size.

    repartition(10).write → 10 files, each ~same size

Cost: full shuffle (all data moves across the network).
Benefit: perfectly even output files.

**Solution**

Use repartition before write when you need EVEN file sizes or need to INCREASE the partition count.

**Key Points**

DO:
  - Use when output files MUST be evenly sized (downstream systems need it).
  - Use to INCREASE file count (coalesce can only decrease).
  - repartition(n, "col") to co-locate same-key rows in same files —
    improves downstream filter/join performance.

DON'T:
  - Use repartition when coalesce would work — repartition is slower
    (full shuffle vs. local merge).
  - Repartition to 1 — same problem as coalesce(1): one giant file.
  - Repartition by a high-cardinality column with low n — hash collisions
    make files uneven anyway.

WHEN TO USE:
  coalesce → reducing file count, unevenness is acceptable.
  repartition → need EXACT count with EVEN sizes, or need to increase.


**Implementation**

```
def demo_repartition_before_write():
    """Repartition creates evenly-sized output files."""

    df = spark.range(0, 100000) \
        .withColumn("category", (F.col("id") % 5).cast("string")) \
        .withColumn("amount", (F.col("id") * 2.0).cast("double"))

    # --- Repartition for even file sizes ---
    df.repartition(10).write.mode("overwrite") \
        .parquet("/tmp/training/write_repartitioned")

    # --- Repartition by column — co-locates same category in same file ---
    df.repartition(5, "category").write.mode("overwrite") \
        .parquet("/tmp/training/write_repartitioned_by_col")

    files = spark.read.parquet("/tmp/training/write_repartitioned").inputFiles()
    print(f"repartition(10): {len(files)} files (evenly sized)")

    files_by_col = spark.read.parquet("/tmp/training/write_repartitioned_by_col").inputFiles()
    print(f"repartition(5, 'category'): {len(files_by_col)} files (grouped by category)")
```

##### 3. Partition By (partitionBy)

**Problem**

A table has 1 TB of data across all dates.  Every query filters by date:
    WHERE date = '2025-01-15'

Without partitioning, Spark reads ALL 1 TB — then throws away 99.7%.

**Concept**

partitionBy("col") writes data into a FOLDER STRUCTURE based on column
values:

    /output/date=2025-01-01/part-00000.parquet
    /output/date=2025-01-02/part-00000.parquet
    /output/date=2025-01-03/part-00000.parquet

When a query filters WHERE date = '2025-01-15', Spark reads ONLY that
folder — skipping all other dates.  This is PARTITION PRUNING (see read
optimizations).

partitionBy is a WRITE-TIME decision that enables READ-TIME optimization.

**Solution**

Partition by the column(s) most frequently used in WHERE filters.

**Key Points**

DO:
  - Partition by LOW-CARDINALITY columns: date, region, status, country.
    Ideal: 100s to low 1000s of distinct values.
  - Partition by the column used in downstream WHERE filters.
  - Combine with coalesce/repartition to control files PER partition:
      df.repartition(1, "date").write.partitionBy("date").parquet(path)
      → 1 file per date partition.

DON'T:
  - Partition by HIGH-CARDINALITY columns (user_id, order_id) —
    millions of folders, each with one tiny file = SMALL FILE PROBLEM.
  - Partition by more than 2-3 columns — combinatorial explosion of folders.
    partitionBy("year", "month", "day", "hour") = way too many folders.
  - Partition by columns that aren't used in filters — write overhead
    with no read benefit.
  - Forget that partition columns are REMOVED from the Parquet file data
    (they're encoded in the folder path).

WHEN TO USE:
  - Any table > 1 GB that is repeatedly filtered by the same column.
  - Time-series data: partition by date or year/month.
  - Multi-tenant data: partition by tenant_id (if low cardinality).

**Implementation**

```
def demo_partition_by():
    """partitionBy creates folder structure for partition pruning."""

    data = [(f"user_{i}", "2025-01-15" if i % 3 == 0 else "2025-01-16", float(i))
            for i in range(10000)]
    df = spark.createDataFrame(data, ["user", "date", "amount"])

    # --- Write partitioned by date ---
    df.write.mode("overwrite") \
        .partitionBy("date") \
        .parquet("/tmp/training/write_partitioned")

    # --- Read with filter → partition pruning ---
    result = spark.read.parquet("/tmp/training/write_partitioned") \
        .filter("date = '2025-01-15'")

    print("=== partitionBy('date') → Partition Pruning on Read ===")
    result.explain(True)
    # Look for: PartitionFilters: [date = 2025-01-15]
    print(f"Rows for 2025-01-15: {result.count()}")

    # --- Control files per partition ---
    df.repartition(1, "date").write.mode("overwrite") \
        .partitionBy("date") \
        .parquet("/tmp/training/write_partitioned_1file")
    print("\nrepartition(1, 'date') + partitionBy('date') → 1 file per date folder")
```

##### **4. Optimize Write (Delta Lake)**

**Problem**

- In Spark, each task writes ONE output file.  If you have 200 tasks but only 50 MB of data, you get 200 files × 250 KB each — tiny files.
- This happens even without explicit repartition/coalesce because the number of output files = number of tasks writing.

**Concept**

optimizeWrite is a Delta Lake feature that automatically COALESCES partitions at write time.  It re-bins data so output files are close to the target size (~128 MB) regardless of how many tasks there are.

    Without optimizeWrite:  200 tasks → 200 files (many tiny)
    With optimizeWrite:     200 tasks → Spark re-bins → ~5 files (right-sized)

It acts like an automatic coalesce INSIDE the write operation.

**Solution**

Enable for Delta tables (automatic in Databricks, manual in OSS Spark).

**Key Points**

DO:
  - Enable for all Delta table writes:
      spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
    Or per-table: ALTER TABLE t SET TBLPROPERTIES (delta.autoOptimize.optimizeWrite = true)
  - Use instead of manual coalesce for Delta tables — it handles sizing
    automatically.
  - Combine with autoCompact for full automation (see section 5).

DON'T:
  - Use with explicit repartition/coalesce — they conflict.  Let
    optimizeWrite handle file sizing automatically.
  - Expect it to work on plain Parquet — Delta Lake feature only.
  - Assume it replaces OPTIMIZE — it only helps at WRITE time;
    existing small files still need OPTIMIZE to compact.

WHEN TO USE:
  - All Delta table writes — no downside.
  - Especially for streaming or frequent small appends.

**Implementation**

```
def demo_optimize_write():
    """optimizeWrite auto-sizes output files at write time (Delta)."""

    # --- Delta Lake required — uncomment if available ---
    # spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")

    # df = spark.range(0, 100000) \
    #     .withColumn("value", (F.col("id") * 2.0).cast("double"))
    #
    # # Without optimizeWrite: many tasks → many small files
    # df.write.mode("overwrite").format("delta") \
    #     .save("/tmp/training/delta_no_optwrite")
    #
    # # With optimizeWrite: tasks auto-coalesced → fewer, right-sized files
    # df.write.mode("overwrite").format("delta") \
    #     .option("optimizeWrite", "true") \
    #     .save("/tmp/training/delta_optwrite")

    print("=== optimizeWrite (Delta Lake) ===")
    print("Enable: spark.conf.set('spark.databricks.delta.optimizeWrite.enabled', 'true')")
    print("Or per-write: .option('optimizeWrite', 'true')")
    print("Effect: auto-coalesces partitions → fewer, right-sized output files")
    print("Use for: all Delta writes, especially streaming/frequent appends")
```

##### **5. Auto Compact (Delta Lake)**

**Problem**

Even with optimizeWrite, repeated APPEND operations accumulate files:

    Day 1: append → 5 files
    Day 2: append → 5 more files (total: 10)
    Day 30: append → 5 more (total: 150 files)

optimizeWrite controls per-write file count.  But across many writes,
files accumulate.  Eventually you have hundreds of small files.

**Concept**

autoCompact triggers a MINI OPTIMIZE automatically after each write.
When the number of small files in a partition exceeds a threshold,
Spark compacts them into larger files.

    After append: check → too many small files? → compact → done

It runs in the SAME job, right after the write — no separate job needed.

autoCompact does a LIGHTER compaction than full OPTIMIZE:
  - Only compacts files < 128 MB
  - Doesn't rewrite large files
  - Doesn't do ZORDER (just file sizing)

**Solution**

Enable for Delta tables with frequent appends.

**Key Points**

DO:
  - Enable with optimizeWrite for full automation:
      spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")
    Or per-table: ALTER TABLE t SET TBLPROPERTIES (delta.autoOptimize.autoCompact = true)
  - Use for streaming sinks and incremental ETL that append frequently.

DON'T:
  - Rely on autoCompact alone for large tables — it does light compaction.
    Still run full OPTIMIZE periodically for best performance (see section 6).
  - Enable on tables that are rarely appended to — no benefit, just overhead.
  - Expect ZORDER from autoCompact — it only sizes files, doesn't reorganize
    data layout.  Use OPTIMIZE ZORDER BY for that.

WHEN TO USE:
  - Tables with frequent appends (streaming, hourly/daily ingestion).
  - Combine: optimizeWrite (per-write) + autoCompact (across writes).

**Implementation**
```
def demo_auto_compact():
    """autoCompact triggers light compaction after each write (Delta)."""

    # --- Delta Lake required ---
    # spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

    # # Simulate frequent appends
    # for batch in range(5):
    #     df = spark.range(batch * 1000, (batch + 1) * 1000) \
    #         .withColumn("value", (F.col("id") * 1.5).cast("double"))
    #     df.write.mode("append").format("delta") \
    #         .save("/tmp/training/delta_autocompact")
    #     # autoCompact checks after each append and compacts if needed

    print("=== autoCompact (Delta Lake) ===")
    print("Enable: spark.conf.set('spark.databricks.delta.autoCompact.enabled', 'true')")
    print("Effect: after each write, compacts small files if too many accumulate")
    print("Light compaction — only merges files < 128 MB, no ZORDER")
    print("Use for: streaming sinks, frequent appends")
    print("Still run full OPTIMIZE periodically for best read performance")
```

##### 6. Optimize (Delta Lake)

**Problem**

Over time, Delta tables accumulate many small files from:
  - Streaming micro-batches (every 30 seconds → hundreds of files/day)
  - Frequent appends (hourly ETL)
  - UPDATE/DELETE/MERGE operations (copy-on-write creates new files)

Reading 10,000 small files is much slower than reading 100 right-sized
files because of:
  - File listing overhead (10,000 LIST calls on cloud storage)
  - Per-file open/close overhead
  - Poor compression (small files don't compress well)
  - Too many tasks in Spark

**Concept**

OPTIMIZE compacts small files into larger, optimally-sized files
(target: ~1 GB per file).

    BEFORE: 10,000 files × 1 MB each = 10 GB
    AFTER:  10 files × 1 GB each = 10 GB  (same data, fewer files)

Process:
  1. Read all small files in a partition
  2. Rewrite them into fewer, larger files
  3. Update the Delta transaction log
  4. Old files are marked for deletion (cleaned up by VACUUM)

OPTIMIZE is IDEMPOTENT — running it twice has no effect if files are
already optimally sized.

**Solution**

Run OPTIMIZE periodically on Delta tables.

**Key Points**

DO:
  - Run OPTIMIZE after bulk data loading completes.
  - Schedule OPTIMIZE as a maintenance job (daily or weekly).
  - Use WHERE clause to optimize specific partitions:
      OPTIMIZE table WHERE date = '2025-01-15'
  - Combine with ZORDER BY for data co-location (see section 7).

DON'T:
  - Run OPTIMIZE during active writes — it competes for resources.
  - OPTIMIZE too frequently on append-heavy tables — each run rewrites
    files; use autoCompact for continuous light compaction.
  - Forget to VACUUM after OPTIMIZE — old files consume storage.

WHEN TO USE:
  - After initial bulk load of a Delta table.
  - Periodically on tables with many small files.
  - Before running analytics on a table with accumulated appends.

**Implementation**
```
def demo_optimize():
    """OPTIMIZE compacts small files into larger ones (Delta)."""

    # --- Delta Lake required ---
    # # Create table with many small files (simulating micro-batches)
    # for i in range(20):
    #     df = spark.range(i * 500, (i + 1) * 500) \
    #         .withColumn("date", F.lit("2025-01-15")) \
    #         .withColumn("value", (F.col("id") * 1.5).cast("double"))
    #     df.write.mode("append").format("delta") \
    #         .partitionBy("date") \
    #         .save("/tmp/training/delta_optimize_demo")
    #
    # # Check file count BEFORE optimize
    # detail_before = spark.sql("DESCRIBE DETAIL delta.`/tmp/training/delta_optimize_demo`")
    # detail_before.select("numFiles").show()  # ~20 small files
    #
    # # --- OPTIMIZE: compact all files ---
    # spark.sql("OPTIMIZE delta.`/tmp/training/delta_optimize_demo`")
    #
    # # --- OPTIMIZE specific partition ---
    # spark.sql("""
    #     OPTIMIZE delta.`/tmp/training/delta_optimize_demo`
    #     WHERE date = '2025-01-15'
    # """)
    #
    # # Check file count AFTER optimize
    # detail_after = spark.sql("DESCRIBE DETAIL delta.`/tmp/training/delta_optimize_demo`")
    # detail_after.select("numFiles").show()  # ~1-2 right-sized files

    print("=== OPTIMIZE (Delta Lake) ===")
    print("Compacts small files into larger ~1 GB files")
    print("")
    print("Usage:")
    print("  OPTIMIZE my_table                          -- full table")
    print("  OPTIMIZE my_table WHERE date = '2025-01-15' -- one partition")
    print("  OPTIMIZE my_table ZORDER BY (col)          -- compact + reorder (see #7)")
    print("")
    print("Schedule: daily/weekly depending on write frequency")
    print("Run AFTER bulk loads, BEFORE analytics")
```

##### **7. ZORDER BY**

**Problem**

OPTIMIZE compacts files but doesn't change how data is ORDERED inside
them.  If customer_id values are randomly distributed across files,
every file's min/max range for customer_id spans the full range:

    File 1: customer_id min=1,    max=99999   ← wide range
    File 2: customer_id min=5,    max=99995   ← wide range
    File 3: customer_id min=10,   max=99998   ← wide range

    Query: WHERE customer_id = 500
    Data skipping checks min/max → ALL files overlap → reads ALL files.

**Concept**

ZORDER BY physically reorganizes data so that rows with SIMILAR values
of the specified column(s) are stored in the SAME files.

    BEFORE ZORDER:
      File 1: customer_id [1 - 99999]      ← overlaps everything
      File 2: customer_id [5 - 99995]
      File 3: customer_id [10 - 99998]

    AFTER ZORDER BY (customer_id):
      File 1: customer_id [1 - 33000]      ← tight, non-overlapping
      File 2: customer_id [33001 - 66000]
      File 3: customer_id [66001 - 99999]

    Query: WHERE customer_id = 500
    Data skipping: only File 1 overlaps → reads 1 file instead of 3.

ZORDER uses a space-filling curve (Z-order curve) that maps multi-
dimensional data into a linear order while preserving locality.  This
means you can ZORDER BY (col_a, col_b) and get good skipping on BOTH.

**Solution**

Run OPTIMIZE with ZORDER BY on columns used in WHERE filters.

**Key Points**

DO:
  - ZORDER BY the 1-2 columns most frequently used in WHERE filters.
  - Combine with OPTIMIZE — ZORDER is an option of OPTIMIZE, not separate.
  - Prefer ZORDER on high-cardinality columns where min/max skipping is
    poor without it (customer_id, user_id, timestamp).
  - Re-run periodically as new data is appended (new files won't be
    ZORDERed until the next OPTIMIZE).

DON'T:
  - ZORDER BY more than 3-4 columns — effectiveness drops sharply.
    The Z-order curve can only preserve locality in a few dimensions.
  - ZORDER BY low-cardinality columns (status, country) — min/max
    skipping already works well on these.
  - ZORDER BY the partition column — it's redundant (partition pruning
    already handles it at folder level).
  - Expect ZORDER to be incremental — it rewrites the ENTIRE table
    (or partition) each time.  Use Liquid Clustering for incremental.

WHEN TO USE:
  - Large Delta tables (> 10 GB) with point lookup or range filter queries.
  - Columns with high cardinality used in WHERE: customer_id, timestamp,
    order_id.
  - NOT needed if the table is already sorted on the filter column.

**Implementation**
```
def demo_zorder():
    """ZORDER BY co-locates similar values for better data skipping (Delta)."""

    # --- Delta Lake required ---
    # df = spark.range(0, 1000000) \
    #     .withColumn("customer_id", (F.col("id") % 100000).cast("int")) \
    #     .withColumn("amount", (F.col("id") * 1.5).cast("double"))
    #
    # df.write.mode("overwrite").format("delta") \
    #     .save("/tmp/training/delta_zorder_demo")
    #
    # # --- BEFORE ZORDER: random data → wide min/max per file ---
    # result_before = spark.read.format("delta") \
    #     .load("/tmp/training/delta_zorder_demo") \
    #     .filter("customer_id = 500")
    # # All files are read — data skipping doesn't help
    #
    # # --- ZORDER BY customer_id ---
    # spark.sql("""
    #     OPTIMIZE delta.`/tmp/training/delta_zorder_demo`
    #     ZORDER BY (customer_id)
    # """)
    #
    # # --- AFTER ZORDER: co-located data → tight min/max → skip most files ---
    # result_after = spark.read.format("delta") \
    #     .load("/tmp/training/delta_zorder_demo") \
    #     .filter("customer_id = 500")
    # # Only 1-2 files read instead of all

    print("=== ZORDER BY (Delta Lake) ===")
    print("Reorganizes data so similar values are in the same files")
    print("")
    print("Usage:")
    print("  OPTIMIZE my_table ZORDER BY (customer_id)")
    print("  OPTIMIZE my_table ZORDER BY (customer_id, order_date)  -- multi-column")
    print("")
    print("Effect on data skipping:")
    print("  Before: File min/max = [1, 99999] → every query reads every file")
    print("  After:  File min/max = [1, 33000] → query skips 2 out of 3 files")
    print("")
    print("Best for: high-cardinality columns in WHERE (customer_id, timestamp)")
    print("Limit to: 1-3 columns (effectiveness drops after that)")
    print("Replaces: manual sorting before write")
```

##### **8. V-ORDER (Microsoft Fabric)**

**Problem**

Standard Parquet files use generic encoding (dictionary, run-length, etc.)
that is good but not optimal for read speed.  In Microsoft Fabric's
Lakehouse, reads go through the Fabric engine which can benefit from
specialized encoding.

**Concept**

V-Order is a WRITE-TIME optimization specific to Microsoft Fabric that
applies special sorting and encoding to Parquet files for faster reads.

What it does:
  1. Sorts data within each row group for better compression
  2. Applies optimized page-level encoding
  3. Creates Parquet files that are 10-50% faster to read in Fabric

The files are still standard Parquet — any engine can read them.  But
Fabric's reader extracts extra speed from the V-Order encoding.

V-Order is AUTOMATIC in Fabric notebooks and pipelines.  In open-source
Spark, enable it manually.

**Solution**

Automatic in Fabric.  Manual in Spark:

**Key Points**

DO:
  - Let Fabric apply V-Order automatically (default in Fabric notebooks).
  - Enable manually in Spark if writing to OneLake:
      .option("parquet.vorder.enabled", "true")
  - Use for any table read frequently in Fabric — free read speedup.

DON'T:
  - Expect V-Order benefits outside Fabric — standard Parquet readers
    won't see the 10-50% speedup (they'll read normally).
  - Use V-Order as a replacement for OPTIMIZE/ZORDER — V-Order is about
    encoding, not file compaction or data co-location.
  - Worry about compatibility — V-Order files are standard Parquet;
    any engine (Databricks, open-source Spark) can read them normally.

WHEN TO USE:
  - Writing data to Fabric OneLake that will be read by Fabric engines.
  - NOT needed if data is never read through Fabric.

**Implementation**
```
def demo_vorder():
    """V-Order applies special Parquet encoding for faster Fabric reads."""

    # --- V-Order in Spark ---
    # df = spark.range(0, 100000) \
    #     .withColumn("value", (F.col("id") * 2.0).cast("double"))
    #
    # # Write with V-Order enabled
    # df.write.mode("overwrite") \
    #     .format("delta") \
    #     .option("parquet.vorder.enabled", "true") \
    #     .save("/tmp/training/vorder_demo")

    print("=== V-Order (Microsoft Fabric) ===")
    print("Automatic in Fabric notebooks — no code change needed")
    print("")
    print("Manual in Spark:")
    print("  df.write.format('delta')")
    print("    .option('parquet.vorder.enabled', 'true')")
    print("    .save(path)")
    print("")
    print("Effect: 10-50% faster reads in Fabric engine")
    print("Files remain standard Parquet — any engine can read them")
    print("Combines with: OPTIMIZE, ZORDER, partitionBy (all orthogonal)")
```

##### **9. VACUUM (Delta Lake)**

**Problem**

Delta Lake operations create NEW files but NEVER delete old ones:

  - OPTIMIZE rewrites small files into large ones → old small files remain
  - UPDATE/DELETE/MERGE use copy-on-write → old versions remain
  - Schema changes → old files remain

Over time, storage grows unbounded:

    Actual data:  10 GB
    Old files:    50 GB (from 30 days of OPTIMIZE, UPDATE, DELETE)
    Total:        60 GB → paying for 50 GB of garbage

Old files are kept for TIME TRAVEL (read historical versions).  But after
the retention period, they're pure waste.

**Concept**

VACUUM removes files that are:
  1. No longer referenced by the current Delta log
  2. Older than the retention period (default: 7 days / 168 hours)

    VACUUM my_table RETAIN 168 HOURS

After VACUUM, you CANNOT time-travel to versions older than the
retention period — those files are permanently deleted.

**Solution**

Run VACUUM periodically after OPTIMIZE.

**Key Points**

DO:
  - Run VACUUM regularly (weekly or after OPTIMIZE).
  - Set retention based on your time-travel needs:
      168 hours (7 days) = default and usually sufficient
  - Check storage usage before and after:
      DESCRIBE DETAIL my_table → sizeInBytes
  - Automate: schedule VACUUM as part of your maintenance pipeline.

DON'T:
  - Set retention to 0 hours — breaks concurrent readers/writers.
    Minimum safe: 7 days (168 hours).
  - VACUUM without understanding time-travel impact — once vacuumed,
    old versions are GONE permanently.
  - Run VACUUM during active long-running queries — they may reference
    old files that get deleted mid-query.
  - Skip VACUUM thinking Delta handles it — Delta NEVER auto-deletes
    old files.  VACUUM is the ONLY way.

WHEN TO USE:
  - After running OPTIMIZE (old uncompacted files need cleanup).
  - On tables with frequent UPDATE/DELETE/MERGE operations.
  - When cloud storage costs are a concern.
  - NOT needed on immutable/append-only tables with no OPTIMIZE.

**Implementation**
```
def demo_vacuum():
    """VACUUM removes old files no longer referenced by Delta log."""

    # --- Delta Lake required ---
    # # Create and modify a table
    # df = spark.range(0, 100000) \
    #     .withColumn("value", (F.col("id") * 1.5).cast("double"))
    # df.write.mode("overwrite").format("delta").save("/tmp/training/delta_vacuum_demo")
    #
    # # Simulate updates (creates old file versions)
    # spark.sql("""
    #     UPDATE delta.`/tmp/training/delta_vacuum_demo`
    #     SET value = value * 2
    #     WHERE id < 1000
    # """)
    #
    # # Run OPTIMIZE (creates new compacted files, old remain)
    # spark.sql("OPTIMIZE delta.`/tmp/training/delta_vacuum_demo`")
    #
    # # Check size BEFORE vacuum
    # spark.sql("DESCRIBE DETAIL delta.`/tmp/training/delta_vacuum_demo`").select("numFiles", "sizeInBytes").show()
    #
    # # --- VACUUM: remove old files ---
    # spark.sql("VACUUM delta.`/tmp/training/delta_vacuum_demo` RETAIN 168 HOURS")
    #
    # # Check size AFTER vacuum
    # spark.sql("DESCRIBE DETAIL delta.`/tmp/training/delta_vacuum_demo`").select("numFiles", "sizeInBytes").show()

    print("=== VACUUM (Delta Lake) ===")
    print("Removes old files no longer referenced by the current version")
    print("")
    print("Usage:")
    print("  VACUUM my_table                    -- default 7-day retention")
    print("  VACUUM my_table RETAIN 168 HOURS   -- explicit 7-day retention")
    print("  VACUUM my_table RETAIN 720 HOURS   -- 30-day retention")
    print("")
    print("WARNING: After VACUUM, time-travel to versions older than")
    print("         the retention period is PERMANENTLY impossible")
    print("")
    print("Maintenance pipeline order:")
    print("  1. Run your ETL (writes/appends/merges)")
    print("  2. OPTIMIZE my_table ZORDER BY (col)  -- compact + reorder")
    print("  3. VACUUM my_table RETAIN 168 HOURS   -- clean up old files")
```

# MEMORY Optimizations
Control how Spark uses executor/driver memory.

| #  | Optimization              | What It Fixes                                         |
|----|---------------------------|-------------------------------------------------------|
| 1 | Caching (`df.cache()`)    | Avoids recomputation of reused DataFrames             |
| 2 | Persistence Levels        | Controls memory vs disk tradeoff for cached data      |
| 3 | `unpersist()`             | Frees memory when cached data is no longer needed     |

##### **1. Caching (df.cache())**

**Problem**

A DataFrame is used multiple times in the same pipeline:

    df = spark.read.parquet("/data/sales")        # read from disk
    total = df.agg(F.sum("amount")).collect()      # reads from disk
    by_region = df.groupBy("region").count()       # reads from disk AGAIN
    filtered = df.filter("amount > 1000").count()  # reads from disk AGAIN

Each action triggers a FULL re-read and re-computation from scratch.
If reading from cloud storage (S3, ADLS, GCS), each re-read means:
  - Network I/O to fetch data again
  - Decompression of Parquet files again
  - Applying all transformations again

For a 10 GB table used 5 times: 50 GB of I/O instead of 10 GB.

**Concept**

cache() stores the DataFrame's computed result IN MEMORY on the
executors.  After the first action materializes the data, subsequent
actions read from memory instead of re-reading from disk/cloud.

    df.cache()           # marks for caching (lazy — nothing happens yet)
    df.count()           # FIRST action: reads from disk, stores in memory
    df.filter(...).show() # SECOND action: reads from MEMORY (fast!)
    df.groupBy(...).count() # THIRD action: reads from MEMORY (fast!)

cache() = persist(StorageLevel.MEMORY_AND_DISK)
  - Tries to store in memory
  - If memory is full, spills remaining partitions to disk
  - Data is stored DESERIALIZED (as Java/Python objects) for fast access

**Solution**

Call cache() on DataFrames that are reused in multiple actions.

**Key Points**

DO:
  - Cache DataFrames used in 2+ actions within the same job/notebook.
  - Call an action (count, show, collect) after cache() to materialize it:
      df.cache()
      df.count()  # triggers actual caching
  - Monitor cache usage in Spark UI → Storage tab.
  - unpersist() when done (see section 3) to free memory.

DON'T:
  - Cache DataFrames used only ONCE — caching adds overhead with no benefit.
  - Cache very large DataFrames that exceed executor memory — causes
    excessive spill to disk, GC pressure, and potential OOM errors.
  - Cache raw/unfiltered DataFrames — cache AFTER filtering to reduce
    the amount of data stored.
  - Assume cache() is immediate — it's LAZY.  Data is cached only after
    the first action.
  - Cache inside a loop that creates new DataFrames each iteration —
    memory fills up with stale cached data.

WHEN TO USE:
  - DataFrame reused in multiple actions (aggregations, joins, writes).
  - Iterative ML algorithms that scan the same data repeatedly.
  - Interactive exploration in notebooks (filter, plot, re-filter).
  - AFTER expensive transformations (joins, aggregations) to avoid
    recomputing them.

**Implementation**
```
def demo_caching():
    """cache() stores DataFrame in memory for reuse across actions."""

    # Create a sample DataFrame
    df = spark.range(0, 1000000) \
        .withColumn("region", (F.col("id") % 5).cast("string")) \
        .withColumn("amount", (F.col("id") * 1.5).cast("double"))

    # --- BAD: Without cache — recomputes from scratch each time ---
    print("=== Without cache() ===")
    import time

    start = time.time()
    count1 = df.filter("amount > 500000").count()
    time1 = time.time() - start

    start = time.time()
    count2 = df.groupBy("region").agg(F.sum("amount")).collect()
    time2 = time.time() - start

    start = time.time()
    count3 = df.filter("region = '1'").count()
    time3 = time.time() - start

    print(f"  Action 1: {time1:.3f}s")
    print(f"  Action 2: {time2:.3f}s")
    print(f"  Action 3: {time3:.3f}s")

    # --- GOOD: With cache — first action caches, rest read from memory ---
    print("\n=== With cache() ===")
    df_cached = df.cache()

    # First action materializes the cache
    start = time.time()
    _ = df_cached.count()  # triggers caching
    cache_time = time.time() - start
    print(f"  Cache materialization: {cache_time:.3f}s")

    start = time.time()
    _ = df_cached.filter("amount > 500000").count()
    time1 = time.time() - start

    start = time.time()
    _ = df_cached.groupBy("region").agg(F.sum("amount")).collect()
    time2 = time.time() - start

    start = time.time()
    _ = df_cached.filter("region = '1'").count()
    time3 = time.time() - start

    print(f"  Action 1 (from cache): {time1:.3f}s")
    print(f"  Action 2 (from cache): {time2:.3f}s")
    print(f"  Action 3 (from cache): {time3:.3f}s")

    # Check cache in Spark UI → Storage tab
    print(f"\n  Is cached: {df_cached.is_cached}")

    # Clean up
    df_cached.unpersist()
```

##### 2. Persistence Levels (persist with StorageLevel)

**Problem**

cache() uses MEMORY_AND_DISK by default — stores deserialized objects
in memory, spills to disk if full.  But this isn't always optimal:

  - Large DataFrames: memory fills up fast → excessive GC, OOM
  - Memory-constrained clusters: not enough RAM for deserialized objects
  - Network-heavy environments: replicated caching wastes bandwidth

You need finer control over WHERE and HOW data is cached.

**Concept**

persist(StorageLevel) gives you control over:

  1. MEMORY vs DISK — where to store
  2. SERIALIZED vs DESERIALIZED — how to store
  3. REPLICATED vs NON-REPLICATED — how many copies

Available StorageLevels:

| StorageLevel              | Memory | Disk | Serialized | Replicas |
|---------------------------|:------:|:----:|:----------:|:--------:|
| MEMORY_ONLY               |   ✓    |      |            |    1     |
| MEMORY_AND_DISK (default) |   ✓    |  ✓   |            |    1     |
| MEMORY_ONLY_SER           |   ✓    |      |     ✓      |    1     |
| MEMORY_AND_DISK_SER       |   ✓    |  ✓   |     ✓      |    1     |
| DISK_ONLY                 |        |  ✓   |     ✓      |    1     |
| MEMORY_ONLY_2             |   ✓    |      |            |    2     |
| MEMORY_AND_DISK_2         |   ✓    |  ✓   |            |    2     |
| OFF_HEAP                  |   ✓*   |      |     ✓      |    1     |

*OFF_HEAP stores outside the JVM heap (avoids GC pressure).*

Key tradeoffs:
  - DESERIALIZED (default): faster access, more memory per object
  - SERIALIZED: slower access (decode needed), ~2-5x less memory usage
  - DISK_ONLY: slowest reads, but no memory pressure at all
  - REPLICATED (_2): fault-tolerant but doubles memory/disk usage

**Solution**

Choose the right StorageLevel based on memory pressure and access pattern.

**Key Points**

DO:
  - Start with cache() (MEMORY_AND_DISK) — good default for most cases.
  - Use MEMORY_ONLY_SER when memory is tight but data is reused often:
      df.persist(StorageLevel.MEMORY_ONLY_SER)
    Saves 2-5x memory at cost of ~10-20% slower access (serialization).
  - Use DISK_ONLY for very large DataFrames reused a few times:
      df.persist(StorageLevel.DISK_ONLY)
    No memory pressure, but reads are slower (disk I/O).
  - Use OFF_HEAP for critical caches that must avoid GC pauses:
      df.persist(StorageLevel.OFF_HEAP)
    Requires: spark.memory.offHeap.enabled=true, spark.memory.offHeap.size

DON'T:
  - Use MEMORY_ONLY on data larger than available executor memory —
    partitions that don't fit are RECOMPUTED on every access (no spill).
  - Use _2 (replicated) unless you're on an unstable cluster with
    frequent executor failures — doubles resource usage.
  - Mix persist levels on the same DataFrame — unpersist first, then
    re-persist with the new level.
  - Forget that persist() is also LAZY — call an action to materialize.

WHEN TO USE:
  MEMORY_AND_DISK (cache):  Default — most use cases.
  MEMORY_ONLY_SER:          Memory-constrained + frequent reuse.
  DISK_ONLY:                Very large data, moderate reuse.
  OFF_HEAP:                 Low-latency requirements, avoid GC pauses.
  _2 (replicated):          Unstable clusters, can't afford recomputation.

**Implementation**

```
def demo_persistence_levels():
    """persist(StorageLevel) gives fine-grained control over caching."""

    df = spark.range(0, 500000) \
        .withColumn("category", (F.col("id") % 10).cast("string")) \
        .withColumn("value", (F.col("id") * 2.5).cast("double"))

    # --- MEMORY_ONLY: fastest reads, partitions lost if memory full ---
    print("=== MEMORY_ONLY ===")
    df_mem = df.persist(StorageLevel.MEMORY_ONLY)
    df_mem.count()  # materialize
    print(f"  Cached: {df_mem.is_cached}")
    print("  ✓ Fastest reads — deserialized in-memory objects")
    print("  ✗ No spill to disk — partitions recomputed if evicted")
    df_mem.unpersist()

    # --- MEMORY_ONLY_SER: smaller memory footprint ---
    print("\n=== MEMORY_ONLY_SER ===")
    df_ser = df.persist(StorageLevel.MEMORY_ONLY_SER)
    df_ser.count()
    print(f"  Cached: {df_ser.is_cached}")
    print("  ✓ 2-5x less memory than MEMORY_ONLY")
    print("  ✗ ~10-20% slower reads (deserialization cost)")
    print("  Best for: memory-constrained clusters")
    df_ser.unpersist()

    # --- MEMORY_AND_DISK: safe default ---
    print("\n=== MEMORY_AND_DISK (same as cache()) ===")
    df_md = df.persist(StorageLevel.MEMORY_AND_DISK)
    df_md.count()
    print(f"  Cached: {df_md.is_cached}")
    print("  ✓ Best of both — memory first, spill to disk")
    print("  ✓ No recomputation — everything is stored somewhere")
    print("  Default and recommended for most cases")
    df_md.unpersist()

    # --- DISK_ONLY: no memory usage ---
    print("\n=== DISK_ONLY ===")
    df_disk = df.persist(StorageLevel.DISK_ONLY)
    df_disk.count()
    print(f"  Cached: {df_disk.is_cached}")
    print("  ✓ Zero memory pressure")
    print("  ✗ Slower reads (disk I/O for every access)")
    print("  Best for: very large DataFrames, moderate reuse")
    df_disk.unpersist()

    # --- MEMORY_AND_DISK_SER: balanced ---
    print("\n=== MEMORY_AND_DISK_SER ===")
    df_mds = df.persist(StorageLevel.MEMORY_AND_DISK_SER)
    df_mds.count()
    print(f"  Cached: {df_mds.is_cached}")
    print("  ✓ Serialized in memory (smaller footprint)")
    print("  ✓ Spills to disk if memory full")
    print("  Best for: large data + memory-constrained + frequent reuse")
    df_mds.unpersist()

    # --- Summary table ---
    print("\n=== Persistence Level Decision Guide ===")
    print("┌─────────────────────────┬─────────────┬───────────┬──────────────┐")
    print("│ Level                   │ Memory Cost │ Read Speed│ Best For     │")
    print("├─────────────────────────┼─────────────┼───────────┼──────────────┤")
    print("│ MEMORY_ONLY             │ High        │ Fastest   │ Small data   │")
    print("│ MEMORY_AND_DISK         │ High        │ Fast      │ Default      │")
    print("│ MEMORY_ONLY_SER         │ Low         │ Fast      │ Large + reuse│")
    print("│ MEMORY_AND_DISK_SER     │ Low         │ Fast      │ Large + safe │")
    print("│ DISK_ONLY               │ Zero        │ Slow      │ Very large   │")
    print("└─────────────────────────┴─────────────┴───────────┴──────────────┘")
```

##### **3. Unpersist (Freeing Cached Data)**

**Problem**

Every cache() call consumes executor memory.  If you cache multiple DataFrames and forget to free them, memory fills up:

    df1.cache(); df1.count()   # 2 GB cached
    df2.cache(); df2.count()   # 3 GB cached
    df3.cache(); df3.count()   # 4 GB cached — total: 9 GB in cache

    # df1 is no longer needed, but still consuming 2 GB
    # New computations get less memory → spill to disk → slower

Worse: Spark's storage memory and execution memory SHARE the same pool.
Cached data that isn't freed steals memory from shuffles, sorts, and
aggregations — making active computations slower.

    ┌──────────────────────────────────────────────────┐
    │              Spark Unified Memory Pool            │
    │                                                  │
    │  ┌──────────────┐  ┌──────────────────────────┐  │
    │  │  Storage     │  │   Execution              │  │
    │  │  (cached DFs)│  │   (shuffles, sorts,      │  │
    │  │  2 GB used   │←→│    aggregations)          │  │
    │  │              │  │   Gets less if storage    │  │
    │  │              │  │   is full                 │  │
    │  └──────────────┘  └──────────────────────────┘  │
    └──────────────────────────────────────────────────┘

**Concept**

unpersist() removes a DataFrame from the cache, freeing the memory
for other computations.

    df.unpersist()           # non-blocking — marks for removal
    df.unpersist(blocking=True)  # blocks until fully removed

After unpersist(), subsequent actions on the DataFrame will recompute
it from scratch (re-read from source, re-apply transformations).

**Solution**

Always unpersist DataFrames when they're no longer needed.

**Key Points**

DO:
  - unpersist as soon as you're done with a cached DataFrame:
      df.cache()
      df.count()           # use it
      result = df.groupBy("col").count()  # use it
      df.unpersist()       # done — free the memory

  - Use try/finally to ensure unpersist even on errors:
      df.cache()
      try:
          process(df)
      finally:
          df.unpersist()

  - Check what's cached: spark.catalog.clearCache() clears everything.
  - Monitor Spark UI → Storage tab to see cached DataFrames and sizes.

DON'T:
  - Leave DataFrames cached across notebook cells "just in case" —
    you'll run out of memory eventually.
  - unpersist a DataFrame that's still being used by downstream
    operations — they'll have to recompute from scratch.
  - Rely on Python garbage collection to unpersist — Spark's JVM cache
    is NOT freed when Python objects go out of scope.
  - Call unpersist() on a DataFrame that was never cached — it's a no-op
    but indicates confused logic.

WHEN TO USE:
  - Immediately after the last action that uses the cached DataFrame.
  - At the end of a processing stage before starting the next one.
  - In cleanup blocks (finally) for error-safe memory management.
  - When switching between large cached DataFrames in a notebook.
"""

**Implementation**
```
def demo_unpersist():
    """unpersist() frees memory when cached data is no longer needed."""

    df1 = spark.range(0, 500000) \
        .withColumn("value", (F.col("id") * 1.5).cast("double"))

    df2 = spark.range(0, 500000) \
        .withColumn("value", (F.col("id") * 2.5).cast("double"))

    # --- Cache both DataFrames ---
    df1.cache()
    df2.cache()

    # Materialize caches
    df1.count()
    df2.count()

    print("=== Before unpersist ===")
    print(f"  df1 cached: {df1.is_cached}")
    print(f"  df2 cached: {df2.is_cached}")

    # --- Process df1, then free it ---
    total1 = df1.agg(F.sum("value")).collect()[0][0]
    print(f"\n  df1 sum: {total1}")

    df1.unpersist()  # done with df1 — free memory
    print(f"\n=== After df1.unpersist() ===")
    print(f"  df1 cached: {df1.is_cached}")  # False
    print(f"  df2 cached: {df2.is_cached}")  # True — still available

    # --- Continue using df2 ---
    total2 = df2.agg(F.sum("value")).collect()[0][0]
    print(f"\n  df2 sum: {total2}")

    df2.unpersist()
    print(f"\n=== After df2.unpersist() ===")
    print(f"  df2 cached: {df2.is_cached}")  # False

    # --- Error-safe pattern ---
    print("\n=== Error-safe caching pattern ===")
    df = spark.range(0, 100000).cache()
    try:
        df.count()
        result = df.agg(F.sum("id")).collect()[0][0]
        print(f"  Result: {result}")
    finally:
        df.unpersist()
        print("  Cache freed in finally block")

    # --- Clear ALL caches ---
    print("\n=== Clear all caches ===")
    spark.catalog.clearCache()
    print("  spark.catalog.clearCache() — all cached DataFrames freed")
```

# RESOURCE Optimizations
Control cluster-level resource allocation.

| #  | Optimization                     | What It Fixes                                      |
|----|----------------------------------|----------------------------------------------------|
| 1 | Dynamic Resource Allocation      | Auto-scales executors up/down based on demand      |

# Azure Synapse Spark — Nodes, Executors & vCores

## Complete Guide with Examples

**Topics Covered:**
vCore Calculation • Node Sizing • Dynamic Executors • Wastage Analysis • Quota Management • Best Practices

**March 2026**

---

## 1. Fundamentals — Nodes, Executors & vCores

### 1.1 What is a Node?

A Node is a Virtual Machine (VM) provisioned by Azure Synapse. It is the physical unit of compute. Every node has a fixed number of vCores and RAM determined by the Node Size family.

> These are **Synapse-specific** node sizes, not general Azure VM SKUs.

| Node Size  | vCores | RAM    | Family           |
|------------|--------|--------|------------------|
| Small      | 4      | 32 GB  | Memory Optimized |
| Medium     | 8      | 64 GB  | Memory Optimized |
| Large      | 16     | 128 GB | Memory Optimized |
| XLarge     | 32     | 256 GB | Memory Optimized |
| XXLarge    | 64     | 432 GB | Memory Optimized |
| XXXLarge   | 128    | 864 GB | Memory Optimized |

### 1.2 What is an Executor?

An Executor is a JVM process that runs on a Node and performs the actual data processing. Executors receive tasks from the Driver and process Spark partitions in parallel.

### 1.3 What is a vCore?

A vCore (virtual core) is a CPU thread. Each vCore processes exactly one Spark task (partition) at a time.

**Key Rule:** `1 vCore = 1 Task = 1 Partition processed simultaneously`

### 1.4 The Relationship

```
Workspace
  └── Spark Pool (pool-level minimum: 3 nodes)
        └── Node (VM)
              └── Executor (JVM Process)
                    ├── vCore 1  →  Task 1  →  Partition 1
                    ├── vCore 2  →  Task 2  →  Partition 2
                    └── vCore N  →  Task N  →  Partition N
```

---

## 2. Dynamic Executors DISABLED — 1 Node = 1 Executor

### 2.1 How it Works

When Dynamic Executors (Dynamically allocate executors) is **DISABLED**, Synapse assigns the entire node to a single executor. This is a strict one-to-one mapping.

```
Node (XXLarge: 64 vCores / 432 GB)
  └── Executor 1
        ├── vCores : 64  (full node)
        └── Memory : 432 GB (full node)
```

> **One-to-One Rule:** Dynamic Executors OFF → 1 Node = 1 Executor = 64 vCores = 432 GB

### 2.2 Example — Default Config (3 Nodes, 2 Executors)

With the Synapse **pool-level** minimum of 3 nodes and 2 default executors:

```
Node 1  →  Driver       (orchestration, no data processing)
Node 2  →  Executor 1   (64 vCores, processes partitions)
Node 3  →  Executor 2   (64 vCores, processes partitions)
```

| Metric          | Value                     |
|-----------------|---------------------------|
| Total vCores    | 3 × 64 = 192 vCores requested |
| Working vCores  | 2 × 64 = 128 vCores (executors only) |
| Parallel Tasks  | 128 tasks at a time       |

| Node   | Role       | vCores | Processes Data?            |
|--------|------------|--------|----------------------------|
| Node 1 | Driver     | 64     | No — orchestration only    |
| Node 2 | Executor 1 | 64     | Yes                        |
| Node 3 | Executor 2 | 64     | Yes                        |

### 2.3 vCore Formula — Dynamic Executors OFF

```
Total vCores = (Executor Nodes + 1 Driver Node) × vCores per Node
             = (Executors + 1) × 64

Example: 5 Executors
  = (5 + 1) × 64
  = 6 × 64
  = 384 vCores
```

### 2.4 Node Count for Common Executor Targets

| Executors | Nodes Needed | Total vCores | Working vCores |
|-----------|-------------|--------------|----------------|
| 2         | 3           | 192          | 128            |
| 4         | 5           | 320          | 256            |
| 5         | 6           | 384          | 320            |
| 10        | 11          | 704          | 640            |
| 20        | 21          | 1344         | 1280           |

### 2.5 Autoscale Behaviour — Dynamic Executors OFF

When Dynamic Executors is **DISABLED**, autoscale **cannot add or remove executor processes dynamically**. The number of executors is fixed at session start based on the explicit executor count (either the default or what was set via `spark.executor.instances` / `%%configure`).

However, **the node count is still determined by how many executors are configured**. If `spark.executor.instances` is set to a high value (e.g., 10), Synapse will provision 11 nodes (10 executors + 1 driver) at session start and hold them for the entire session — it will **not scale down** during idle periods or **scale up** mid-session in response to workload.

```
Autoscale: Min 3, Max 20 — Dynamic Executors OFF

Session starts with spark.executor.instances = 2
  → 3 nodes provisioned (min)
  → stays at 3 nodes for entire session
  → Max Nodes = 20 is NEVER reached via autoscale

Session starts with spark.executor.instances = 10
  → 11 nodes provisioned at start (10 executors + 1 driver)
  → stays at 11 nodes for entire session
  → autoscale does NOT scale this down during idle
```

> **Key Point:** With Dynamic Executors OFF, the node count is locked for the session. Autoscale's Min/Max range only acts as a ceiling — it does **not** dynamically adjust nodes mid-session. True elastic scaling requires Dynamic Executors ON.

---

## 3. Dynamic Executors ENABLED — Multiple Executors per Node

### 3.1 How it Works

When Dynamic Executors is **ENABLED**, Synapse packs multiple smaller executors into each node. For XXLarge nodes, Synapse **by default** fits 4 executors per node, each receiving a quarter of the node's resources.

```
Node (XXLarge: 64 vCores / 432 GB) — Dynamic Executors ON
  ├── Executor 1  →  16 vCores / 108 GB
  ├── Executor 2  →  16 vCores / 108 GB
  ├── Executor 3  →  16 vCores / 108 GB
  └── Executor 4  →  16 vCores / 108 GB
```

```
1 Node = 4 Executors (default for XXLarge)
vCores per Executor = 64 ÷ 4 = 16
```

> **Note:** The 4-executors-per-node ratio is the Synapse default for XXLarge. The actual count may vary if `spark.executor.cores` or `spark.executor.memory` are overridden in `%%configure` or pool settings.

**Key Difference:** Dynamic Executors ON → vCores requested = Nodes × 64, NOT Executors × 64

### 3.2 vCore Formula — Dynamic Executors ON

```
vCores per Executor = Node vCores ÷ Executors per Node
                    = 64 ÷ 4 = 16

Total vCores = Nodes × 64  (always based on nodes, not executors)

Example: 8 Executors
  Nodes needed = (8 ÷ 4) + 1 driver = 3 nodes
  Total vCores = 3 × 64 = 192 vCores
  NOT 8 × 64 = 512 (this is WRONG with Dynamic Executors ON)
```

### 3.3 Comparison: Dynamic Executors OFF vs ON

| Setting                  | Dyn Exec OFF          | Dyn Exec ON              |
|--------------------------|-----------------------|--------------------------|
| Mapping                  | 1 Node = 1 Executor   | 1 Node = 4 Executors*    |
| vCores per Executor      | 64 (full node)        | 16 (quarter node)*       |
| RAM per Executor         | 432 GB                | 108 GB*                  |
| 8 Executors → Nodes      | 9 nodes               | 3 nodes                  |
| 8 Executors → vCores     | 9 × 64 = 576          | 3 × 64 = 192             |
| Autoscale works?         | No                    | Yes                      |
| Fault tolerance          | Lower                 | Higher                   |
| Resource efficiency      | Lower                 | Higher                   |

*\*Default for XXLarge. May vary if executor core/memory settings are customized.*

### 3.4 Perfect Executor Count — Fill All Nodes

To avoid wasting vCores on partially filled nodes, always set Max Executors as a **multiple of 4** (for XXLarge nodes at default settings).

```
Min Nodes = 3 (Synapse pool-level default)
Executor Nodes = 3 - 1 driver = 2 nodes
Max Executors  = 2 × 4 = 8  ← fills all nodes perfectly

Node 1  →  Driver
Node 2  →  Executor 1 + 2 + 3 + 4  (full)
Node 3  →  Executor 5 + 6 + 7 + 8  (full)

Total vCores = 3 × 64 = 192
Wasted vCores = 0
```

| Max Executors | Nodes Used | Node 3 Usage         | vCores Wasted |
|---------------|-----------|----------------------|---------------|
| 4             | 3         | Idle — wasted        | 64 vCores     |
| 5             | 3         | 25% used (1 of 4)    | 48 vCores     |
| 6             | 3         | 50% used (2 of 4)    | 32 vCores     |
| 7             | 3         | 75% used (3 of 4)    | 16 vCores     |
| 8             | 3         | 100% full            | 0 — perfect   |

> **Best Practice:** For XXLarge nodes (default config), always set Max Executors as a multiple of 4 (4, 8, 12, 16...) to avoid partial node wastage.

---

## 4. vCore Calculation & Workspace Quota

### 4.1 Your Pool Config (from screenshot)

| Setting            | Value                              |
|--------------------|------------------------------------|
| Node Size Family   | Memory Optimized                   |
| Node Size          | XXLarge (64 vCores / 432 GB)       |
| Autoscale          | Enabled (Min 3, Max 20)            |
| Dynamic Executors  | DISABLED                           |
| Intelligent Cache  | 50%                                |
| Workspace Quota    | 2000 vCores                        |
| Available vCores   | 416 vCores (from quota error)      |

### 4.2 Why the CAPACITY_EXCEEDED Error Occurred

```
Error: Your job requested 704 vCores
       Only 416 available
```

**Breakdown:**

```
704 = 11 nodes × 64 vCores
    = 10 executor nodes + 1 driver
```

**Root Cause:** With Dynamic Executors OFF, `spark.executor.instances` was set to 10 (either explicitly via `%%configure`, pipeline config, or a parent notebook call). Since each executor needs its own dedicated node, Synapse provisioned 11 nodes (10 executors + 1 driver) at session start — all 704 vCores at once.

Meanwhile, the workspace already had other pools consuming quota:

```
2000 - 416 = 1584 vCores used by other pools/sessions
Only 416 remained → 704 > 416 → FAILED
```

> **Fix:** Either reduce executor count to fit within available quota, cap Max Nodes, or enable Dynamic Executors so that 10 executors fit on fewer nodes (3 nodes = 192 vCores instead of 704).

### 4.3 Safe Node Calculation for Your Workspace

```
Available vCores        = 416
vCores per Node         = 64

Max safe nodes          = 416 ÷ 64 = 6.5 → 6 nodes
Max executor nodes      = 6 - 1 driver  = 5
Max safe executors      = 5  (Dyn OFF) or 20 (Dyn ON, 4 per node)

Safe vCores used        = 6 × 64 = 384
Remaining for others    = 416 - 384 = 32 vCores headroom
```

### 4.4 Recommended Config for Your Pool

| Setting          | Current       | Recommended                     |
|------------------|---------------|---------------------------------|
| Min Nodes        | 3             | 3                               |
| Max Nodes        | 20            | 6                               |
| Dynamic Executors| Disabled      | **Enable** (Min 2, Max 8)       |
| Max Executors    | N/A           | 8 (multiple of 4)               |
| vCores at start  | 192           | 192 (3 nodes × 64)              |
| vCores at max    | 1344          | 384 (6 nodes × 64)              |
| Fits 416 quota?  | No            | **Yes**                         |
| Parallel Tasks   | 128           | 128 (same!)                     |

---

## 5. Wastage Analysis

### 5.1 Types of Wastage

| Wastage Type       | Cause                                     | Avoidable? | Fix                              |
|--------------------|-------------------------------------------|------------|----------------------------------|
| Driver node        | Full node reserved for orchestration      | No         | Unavoidable in Spark             |
| Idle nodes         | Nodes provisioned but no executor         | Yes        | Set Max Nodes = Executors + 1    |
| Partial node       | Executors don't fill all node slots       | Yes        | Use multiples of 4               |
| Last batch         | Final batch has fewer tasks than vCores   | Mostly     | Align partition count            |
| Between stages     | All executors idle during shuffle         | Yes        | Enable Dynamic Executors         |
| Session idle       | Notebook not running but pool is ON       | Yes        | Reduce auto-pause to 5 min      |
| Max Nodes unused   | Max set high but never reached            | N/A        | No cost — nodes not provisioned  |

### 5.2 Example — Nodes=3, Executors=2, Dynamic OFF

```
Node 1  →  Driver      →  useful (orchestration)
Node 2  →  Executor 1  →  useful (64 vCores processing)
Node 3  →  Executor 2  →  useful (64 vCores processing)
```

| Metric          | Value                        |
|-----------------|------------------------------|
| Wasted nodes    | 0                            |
| Wasted vCores   | 0 (executor nodes fully used)|
| Driver overhead | 64 vCores (unavoidable)      |

> **Note:** Max Nodes setting does NOT cause wastage. Unprovisioned nodes cost nothing. Only provisioned nodes are billed.

### 5.3 Example — Max Executors=5, Dynamic ON, Nodes=3

```
Node 1  →  Driver
Node 2  →  Executor 1 + 2 + 3 + 4  (4 slots, full)
Node 3  →  Executor 5               (1 of 4 slots used)
           Empty slot  ← 16 vCores wasted
           Empty slot  ← 16 vCores wasted
           Empty slot  ← 16 vCores wasted
```

| Metric              | Value              |
|---------------------|--------------------|
| Node 3 utilization  | 25%                |
| vCores wasted       | 48 out of 64       |
| Fix                 | Set Max Executors = 8 to fill Node 3 |

### 5.4 Quota Blocking vs Billing

These are two different things that are often confused:

| Aspect    | Quota Blocking                         | Billing                         |
|-----------|----------------------------------------|---------------------------------|
| Based on  | Nodes provisioned at session start     | Actual nodes running over time  |
| At start  | All provisioned nodes block quota      | All provisioned nodes billed    |
| Scale-up  | Additional nodes block more quota      | Additional nodes add cost       |
| Scale-down| Released nodes free quota              | Reduced cost                    |

> **Nuance with Dynamic Allocation ON:** Synapse may initially reserve quota for only the min executors' nodes and acquire additional quota as it scales up. If the workspace is near capacity, scale-up may **fail mid-job** rather than failing at session start. Plan headroom accordingly.

---

## 6. Partition & Task Planning

### 6.1 How Tasks Map to vCores

```
Total vCores (executors) = 2 × 64 = 128 (Dyn OFF, 2 executors)
Total tasks              = 2000

Batch 1  : Tasks    1–128   →  128 vCores busy  (full)
Batch 2  : Tasks  129–256   →  128 vCores busy  (full)
...
Batch 15 : Tasks 1793–1920  →  128 vCores busy  (full)
Batch 16 : Tasks 1921–2000  →   80 vCores busy  (partial — 48 idle!)
```

### 6.2 Ideal Partition Count Formula

```
Rule 1 (vCore based):
  ideal_partitions = total_executor_vCores × 2
  Example: 128 vCores × 2 = 256 partitions

Rule 2 (data size based):
  ideal_partitions = total_data_size_MB ÷ 128
  Example: 500 GB = 500,000 MB ÷ 128 = ~3906 partitions

Use the HIGHER of both values.
```

### 6.3 Setting Shuffle Partitions

```python
# Option 1: Set fixed partitions matched to executor vCores
spark.conf.set('spark.sql.shuffle.partitions', '256')

# Option 2 (Recommended): Use AQE to auto-tune at runtime
spark.conf.set('spark.sql.adaptive.enabled', 'true')
spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled', 'true')
# Set a high initial value — AQE will coalesce small partitions automatically
spark.conf.set('spark.sql.shuffle.partitions', '200')
```

> **Note:** With AQE enabled, Spark dynamically coalesces partitions at runtime. You do not need to set a precise value — AQE adjusts based on actual data sizes during execution.

### 6.4 Align Partitions to Avoid Last Batch Waste

```
Executor vCores = 128
Tasks = 2000

2000 ÷ 128 = 15.625  →  last batch is partial (waste!)

Fix: Round up to nearest multiple of 128
  128 × 16 = 2048 partitions
  2048 ÷ 128 = 16 perfect batches, zero last-batch waste
```

```python
spark.conf.set('spark.sql.shuffle.partitions', '2048')
```

---

## 7. Best Practices & Recommended Config

### 7.1 Optimal Pool Settings for Your Workspace

| Setting           | Value       | Reason                              |
|-------------------|-------------|-------------------------------------|
| Node Size         | XXLarge     | Existing — fits use case            |
| Min Nodes         | 3           | Synapse pool-level minimum          |
| Max Nodes         | 6           | Stays within 416 vCore limit        |
| Dynamic Executors | **ON**      | Efficient resource use + autoscale  |
| Min Executors     | 2           | Minimum working config              |
| Max Executors     | 8           | Multiple of 4, fills 2 executor nodes |
| Auto-pause        | 5–10 min    | Avoid idle billing                  |
| Intelligent Cache | 50%         | Keep existing setting               |

### 7.2 Recommended Spark Config (PySpark)

```python
# At top of every notebook
spark.conf.set('spark.sql.adaptive.enabled',                    'true')
spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled', 'true')
spark.conf.set('spark.sql.adaptive.skewJoin.enabled',           'true')
spark.conf.set('spark.sql.shuffle.partitions',                  '256')
spark.conf.set('spark.dynamicAllocation.executorIdleTimeout',   '30s')
```

### 7.3 Golden Rules

1. **1 vCore = 1 Task = 1 Partition** processed at a time
2. **Total vCores = Nodes × vCores per Node** (always based on nodes)
3. **Dynamic Executors OFF** → 1 Node = 1 Executor (one-to-one)
4. **Dynamic Executors ON** → 1 Node = 4 Executors (default for XXLarge)
5. **Max Nodes = Desired Executors + 1 Driver** (when Dyn OFF)
6. **Max Executors = multiple of 4** to avoid partial node waste (when Dyn ON, XXLarge default)
7. **Driver node is unavoidable** — always costs 1 full node
8. **Unprovisioned nodes cost nothing** — Max Nodes is just a ceiling
9. **Autoscale only works when Dynamic Executors is ON**
10. **Align shuffle partitions** to executor vCores × 2

### 7.4 Quick Decision Guide

| Scenario                          | Action                                         |
|-----------------------------------|-------------------------------------------------|
| Getting CAPACITY_EXCEEDED         | Reduce Max Nodes or enable Dynamic Executors    |
| Nodes not scaling up              | Enable Dynamic Executors                        |
| Partial node waste                | Set Max Executors = multiple of 4               |
| High idle cost between stages     | Enable Dynamic Executors                        |
| Session idle billing              | Reduce auto-pause to 5 minutes                  |
| Need more parallelism             | Increase Max Executors (multiple of 4)          |
| Need to protect workspace quota   | Reduce Max Nodes                                |

---

## Summary

For your XXLarge pool with **416 available vCores**: Enable Dynamic Executors (Min 2, Max 8), set Max Nodes = 6. This gives zero partial-node wastage, 192 total vCores, 128 parallel tasks, and stays safely within quota.
